# Step 1: Data Preprocessing and Exploratory Behavioral Analysis

In [ ]:
# -*- coding: utf-8 -*-
"""
=============================================================================
Script Name: HDDM Data Preprocessing and Exploratory Behavioral Analysis (Step 1)
Description: 
  - Filters and cleans trial-level data for HDDM (Informative Priors).
  - Explicitly standardizes emotion legacy coding ('enj' -> 'rew').
  - Implements Response Coding Audit to ensure data integrity.
  - Generates Fair-Ceiling Diagnostics focusing on Rejection Rates.
  - Prepares the final unconstrained matrix for hierarchical modeling.
=============================================================================
"""

import os
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches
from matplotlib.ticker import FuncFormatter, PercentFormatter

# =============================================================================
# GLOBAL CONSTANTS & AESTHETICS (EDA Specific)
# =============================================================================
sns.set_theme(style="ticks", palette="colorblind")
plt.rcParams.update({
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'legend.frameon': False,
    'pdf.fonttype': 42
})

ACC_COLOR = "#004D40"  
REJ_COLOR = "#4A148C"
ACC_ALPHA = 0.52
REJ_ALPHA = 0.75

# Reaction Time (RT) boundaries in milliseconds
# Adjusted minimum RT into 300ms to accommodate complex social cognition 
# processing time required in the Ultimatum Game.
RT_MIN_MS = 300
RT_MAX_MS = 3000

# Behavioral response coding (Original Data)
RESPONSE_ACCEPT = 1
RESPONSE_REJECT = 2
RESPONSE_NONE = 0

# HDDM response coding boundary
HDDM_ACCEPT = 1
HDDM_REJECT = 0

# =============================================================================
# LOGGING SETUP
# =============================================================================
def setup_logger(log_file: str = "step1_data_preparation.log") -> logging.Logger:
    """Initialize logging configuration for process tracking."""
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s %(levelname)s: %(message)s',
        handlers=[
            logging.FileHandler(log_file, encoding='utf-8'),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger(__name__)

# =============================================================================
# MODULE 1: RESPONSE CODING AUDIT
# =============================================================================
def audit_response_coding(df: pd.DataFrame, logger: logging.Logger) -> pd.DataFrame:
    """Strictly audits the mapping between original button presses and HDDM boundaries."""
    logger.info("--- Executing Response Coding Audit ---")
    
    orig_accept = (df['reaction'] == RESPONSE_ACCEPT).sum()
    orig_reject = (df['reaction'] == RESPONSE_REJECT).sum()
    
    response_mapping = {RESPONSE_ACCEPT: HDDM_ACCEPT, RESPONSE_REJECT: HDDM_REJECT}
    df['response_hddm'] = df['reaction'].map(response_mapping)
    
    hddm_accept = (df['response_hddm'] == HDDM_ACCEPT).sum()
    hddm_reject = (df['response_hddm'] == HDDM_REJECT).sum()
    
    if orig_accept != hddm_accept or orig_reject != hddm_reject:
        raise ValueError("CRITICAL: Response mapping mismatch detected!")
    
    if df['response_hddm'].isnull().any():
        unmapped = df.loc[df['response_hddm'].isnull(), 'reaction'].unique()
        raise ValueError(f"Unmapped response values detected: {unmapped}")
        
    audit_data = [{
        'Original_Response': 'Accept (1)', 'Original_Count': orig_accept,
        'HDDM_Boundary': 'Upper (1)', 'HDDM_Count': hddm_accept
    }, {
        'Original_Response': 'Reject (2)', 'Original_Count': orig_reject,
        'HDDM_Boundary': 'Lower (0)', 'HDDM_Count': hddm_reject
    }]
    
    pd.DataFrame(audit_data).to_csv("response_coding_audit.csv", index=False)
    logger.info("Response coding audit passed and exported to 'response_coding_audit.csv'.")
    return df

# =============================================================================
# MODULE 2: FAIR-CEILING DIAGNOSTICS & ROBUSTNESS CHECKS
# =============================================================================
def generate_fair_ceiling_diagnostics(df_valid: pd.DataFrame, logger: logging.Logger):
    """Calculates rejection rates across Fair and Unfair conditions to justify targeting Unfair trials."""
    logger.info("--- Generating Fair-Ceiling Diagnostics ---")
    
    df_valid = df_valid.copy()
    df_valid['Condition_Type'] = np.where(df_valid['Offers_You'] <= 2, 'Unfair (9:1, 8:2)', 
                                 np.where(df_valid['Offers_You'] >= 4, 'Fair (5:5, 6:4)', 'Intermediate'))
    
    df_target = df_valid[df_valid['Condition_Type'].isin(['Unfair (9:1, 8:2)', 'Fair (5:5, 6:4)'])]
    
    summary = df_target.groupby(['Condition_Type', 'emotion']).apply(
        lambda x: pd.Series({
            'Total_Trials': len(x),
            'Rejection_Rate': (x['reaction'] == RESPONSE_REJECT).mean(),
            'RT_Mean': x['RT'].mean() / 1000.0
        })
    ).reset_index()
    
    summary.to_csv("fair_ceiling_diagnostics.csv", index=False)
    logger.info("Fair-ceiling behavior summary exported to 'fair_ceiling_diagnostics.csv'.")
    
    # Simple Visual Verification
    plt.figure(figsize=(8, 5))
    sns.barplot(
        data=summary, x='emotion', y='Rejection_Rate', hue='Condition_Type',
        palette=['#4A148C', '#900C3F'], alpha=0.85
    )
    plt.title("Empirical Rejection Rates: Fair vs. Unfair Offers", pad=15)
    plt.ylabel("Probability of Rejection")
    plt.xlabel("Emotion Condition")
    plt.gca().yaxis.set_major_formatter(PercentFormatter(1.0))
    plt.ylim(0, 1.05)
    plt.legend(title="Offer Type", loc='upper left', bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.savefig("fair_unfair_rejection_rate.pdf", dpi=300)
    plt.close()

def generate_offer_ratio_robustness(df_unfair: pd.DataFrame, logger: logging.Logger):
    """Validates whether 9:1 and 8:2 ratios behave similarly enough to merge."""
    logger.info("--- Generating 9:1 vs 8:2 Robustness Check ---")
    
    summary = df_unfair.groupby(['Offers_You', 'emotion']).apply(
        lambda x: pd.Series({
            'Total_Trials': len(x),
            'Rejection_Rate': (x['reaction'] == RESPONSE_REJECT).mean(),
            'RT_Mean': x['RT'].mean() / 1000.0
        })
    ).reset_index()
    
    summary['Offer_Ratio'] = summary['Offers_You'].map({1: '9:1', 2: '8:2'})
    summary.to_csv("unfair_offer_ratio_behavior_summary.csv", index=False)
    logger.info("Offer ratio robustness check exported to 'unfair_offer_ratio_behavior_summary.csv'.")

# =============================================================================
# DATA LOADING, FILTERING AND HDDM INGESTION
# =============================================================================
def load_and_filter_data(filepath: str, logger: logging.Logger) -> pd.DataFrame:
    """Loads raw data, applies exclusion criteria, and standardizes labels."""
    logger.info(f"Loading raw data from {filepath}")
    
    df = pd.read_csv(filepath)
    logger.info(f"Initial raw data dimensions: {df.shape}")

    exclusion_log = [{'Stage': 'Total_Initial_Trials', 'Count': len(df)}]

    # 1. Standardize emotion legacy labels EARLY: 'enj' -> 'rew'
    df['emotion'] = df['emotion'].astype(str).str.strip().replace({'enj': 'rew'})

    # 2. Missing responses
    omitted_mask = df['reaction'] == RESPONSE_NONE
    df_valid_resp = df[~omitted_mask]
    exclusion_log.append({'Stage': 'Omitted_Responses', 'Count': omitted_mask.sum()})

    # 3. RT boundaries
    fast_mask = df_valid_resp['RT'] < RT_MIN_MS
    df_valid_rt_low = df_valid_resp[~fast_mask]
    exclusion_log.append({'Stage': 'RT_Too_Fast', 'Count': fast_mask.sum()})

    slow_mask = df_valid_rt_low['RT'] > RT_MAX_MS
    df_valid_rt = df_valid_rt_low[~slow_mask]
    exclusion_log.append({'Stage': 'RT_Too_Slow', 'Count': slow_mask.sum()})

    # Trigger diagnostic before filtering fairness
    generate_fair_ceiling_diagnostics(df_valid_rt, logger)

    # 4. Target Unfair Condition Selection (Offers_You == 1 or 2)
    fair_mask = ~df_valid_rt['Offers_You'].isin([1, 2])
    df_unfair = df_valid_rt[~fair_mask].copy()
    exclusion_log.append({'Stage': 'Non_Unfair_Offers_Excluded', 'Count': fair_mask.sum()})
    exclusion_log.append({'Stage': 'Final_Retained_Unfair_Trials', 'Count': len(df_unfair)})
    
    pd.DataFrame(exclusion_log).to_csv("exclusion_summary_flow.csv", index=False)
    logger.info(f"Retained trials (Unfair conditions only): {len(df_unfair)}")
    
    df_unfair = audit_response_coding(df_unfair, logger)
    generate_offer_ratio_robustness(df_unfair, logger)

    return df_unfair

def prepare_hddm_data(df: pd.DataFrame, logger: logging.Logger) -> pd.DataFrame:
    """Constructs the canonical data matrix required for HDDM estimation."""
    df = df.copy()
    
    unique_ids = df['participant_id'].unique()
    id_map = {orig_id: idx for idx, orig_id in enumerate(unique_ids)}
    df['subj_idx'] = df['participant_id'].map(id_map)
    
    pd.DataFrame(list(id_map.items()), columns=['Original_participant_id', 'HDDM_subj_idx']).to_csv('subject_mapping.csv', index=False)

    hddm_df = pd.DataFrame({
        'subj_idx': df['subj_idx'],
        'rt': df['RT'] / 1000.0,
        'response': df['response_hddm'],
        'emotion': df['emotion'],
        'offer_amount': df['Offers_You']
    })

    return hddm_df

# =============================================================================
# PIPELINE EXECUTION
# =============================================================================
def main():
    logger = setup_logger()
    logger.info("="*60)
    logger.info("HDDM DATA PREPARATION PIPELINE INITIATED (Informative Priors)")
    logger.info("="*60)
    
    try:
        input_file = 'trials.csv'
        df_unfair = load_and_filter_data(input_file, logger)
        hddm_df = prepare_hddm_data(df_unfair, logger)
        
        output_file = 'hddm_data_unfair.csv'
        hddm_df.to_csv(output_file, index=False)
        logger.info(f"Final analytical dataset committed to {output_file}")
        
        # Verify emotion categories successfully updated
        emotions_present = hddm_df['emotion'].unique()
        logger.info(f"Emotions preserved for modeling: {emotions_present}")
        
    except Exception as e:
        logger.error(f"Fatal error encountered: {e}")
        raise

if __name__ == "__main__":
    main()

# Step 2a: Global Configuration

In [ ]:
# -*- coding: utf-8 -*-
"""
=============================================================================
STEP 2a: Global Configuration & Cryptographic Lineage (ArviZ-Centric SSOT)
=============================================================================
Description:
  - Establishes the Single Source of Truth (SSOT) for the entire HDDM
    analytical pipeline, designed for the dockerHDDM ArviZ-centric workflow
    (Pan et al., 2025).
  - Four-parameter DDM (v, a, t, z) without across-trial variability
    (sv, st, sz), following Lerche & Voss (2016) and Boehm et al. (2018)
    recommendations for per-condition trial counts of 30-40.
  - Implements stratified convergence criteria (Vehtari et al., 2021):
    Group-level focal parameters require strict bulk/tail ESS for stable
    95% HDI, while subject-level nuisance parameters use relaxed criteria.
  - Dynamically computes MCMC iteration targets based on inferred model
    complexity tiers (simple, medium, complex).
  - Serializes configuration to disk with SHA-256 cryptographic hashing
    for downstream data lineage enforcement.
  - PPC adequacy thresholds for absolute goodness-of-fit evaluation.
=============================================================================
"""

import os
import re
import json
import hashlib
from datetime import datetime
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Any, Union

# =============================================================================
# SHARED UTILITY: Lineage Validation (used by all downstream Steps)
# =============================================================================
def validate_pipeline_lineage(paths: Dict[str, Path], logger=None) -> str:
    """
    Cross-validates the active configuration hash against upstream artifacts.
    Returns the verified hash string, or raises RuntimeError on mismatch.

    This function is importable by all downstream Steps to avoid code
    duplication of lineage validation logic.
    """
    fingerprint_path = paths['manifests'] / "config_fingerprint.json"
    if not fingerprint_path.exists():
        raise FileNotFoundError(
            "Missing configuration fingerprint. Execute Step 2a first."
        )

    with open(fingerprint_path, 'r', encoding='utf-8') as f:
        active_hash = json.load(f).get('config_hash', '')

    if logger:
        logger.info(f"Lineage hash verified: {active_hash[:16]}...")

    return active_hash


def identify_winning_model(paths: Dict[str, Path]) -> str:
    """
    Parses the Step 4 audit trail to extract the winning model name.
    Supports both audit table formats with defensive boolean deserialization.
    """
    import pandas as pd

    # Try primary audit path, then fallback
    audit_path = paths['audit'] / "final_model_selection_audit.csv"
    if not audit_path.exists():
        audit_path = paths['tables_main'] / "model_comparison_summary.csv"
    if not audit_path.exists():
        raise FileNotFoundError("No audit manifest found. Run Steps 2a-4.")

    df = pd.read_csv(audit_path)
    if df.empty:
        raise RuntimeError("Audit log is empty.")

    # Format A: 'model_name' column (GitHub-style audit)
    if 'model_name' in df.columns:
        return str(df['model_name'].iloc[0]).lower()

    # Format B: 'Model' + 'Is_Winner' columns (comparison summary)
    if 'Model' in df.columns and 'Is_Winner' in df.columns:
        mask = df["Is_Winner"].astype(str).str.strip().str.lower() == "true"
        winners = df[mask]
        if winners.empty:
            raise ValueError("No winning model flagged in summary.")
        return str(winners.iloc[0]["Model"]).lower()

    raise KeyError("Audit table lacks expected columns.")


# =============================================================================
# CORE CONFIGURATION DATACLASS
# =============================================================================
@dataclass
class HDDMConfig:
    """
    Analytical configuration schema for the dockerHDDM ArviZ-centric
    pipeline. Manages MCMC hyperparameters, convergence criteria, model
    architecture definitions, and PPC adequacy thresholds.
    """

    # -------------------------------------------------------------------------
    # 1. OPERATIONAL MODE & ROUTING
    # -------------------------------------------------------------------------
    run_mode: str = 'final'
    base_dir: Path = Path(os.getcwd())

    # -------------------------------------------------------------------------
    # 2. STRUCTURAL MODELING CONSTRAINTS
    # -------------------------------------------------------------------------
    # Four-parameter DDM: v, a, t are estimated by default in HDDM;
    # z (starting point bias) requires explicit inclusion.
    # sv, st, sz excluded per Lerche & Voss (2016) given 30-40 trials/cond.
    include_params: List[str] = field(default_factory=lambda: ['z'])

    group_only_regressors: bool = True
    keep_regressor_trace: bool = True    # Required for PPC in HDDMRegressor
    p_outlier: float = 0.05
    use_informative_priors: bool = True
    baseline_condition: str = 'neu'

    # -------------------------------------------------------------------------
    # 3. STRATIFIED MCMC DIAGNOSTIC THRESHOLDS (Vehtari et al., 2021)
    # -------------------------------------------------------------------------
    # Focal Parameters (Group-level Intercepts and Treatment contrasts):
    #   Require strict ESS for stable 95% HDI estimation.
    rhat_focal: float = 1.01
    ess_bulk_focal: float = 1000.0
    ess_tail_focal: float = 500.0

    # Nuisance Parameters (Subject-level deviations, transformed scales):
    #   Relaxed criteria acceptable for hierarchical shrinkage parameters.
    rhat_nuisance: float = 1.05
    ess_bulk_nuisance: float = 400.0
    ess_tail_nuisance: float = 200.0

    # -------------------------------------------------------------------------
    # 4. EXPERIMENTAL DESIGN & VISUAL STANDARDS (Excluded from hash)
    # -------------------------------------------------------------------------
    emotion_order: List[str] = field(
        default_factory=lambda: ['neu', 'rew', 'aff', 'dom', 'dis']
    )

    display_labels: Dict[str, str] = field(default_factory=lambda: {
        'neu': 'Neutral', 'rew': 'Reward', 'aff': 'Affiliative',
        'dom': 'Dominance', 'dis': 'Disgust'
    })

    # Okabe-Ito inspired colorblind-friendly palette
    colors: Dict[str, str] = field(default_factory=lambda: {
        'neu': "#8491B4", 'rew': "#3C5488", 'aff': "#91D1C2",
        'dom': "#F39B7F", 'dis': "#E64B35"
    })

    # -------------------------------------------------------------------------
    # 5. BASE MCMC HYPERPARAMETERS (dockerHDDM ArviZ-centric)
    # -------------------------------------------------------------------------
    n_chains: int = 2 if run_mode == 'debug' else 4
    base_seed: int = 2508
    default_thin: int = 1   # Retain maximal information from Metropolis-Hastings

    # dockerHDDM-specific sampling flags
    # loglike=True enables WAIC/LOO-CV; set False if memory-constrained
    enable_loglike: bool = True
    enable_ppc: bool = True
    ppc_samples: int = 500   # Posterior predictive samples per observed trial

    # Adaptive sampling limits
    max_adaptive_cycles: int = 1 if run_mode == 'debug' else 5

    # -------------------------------------------------------------------------
    # 6. PPC ADEQUACY THRESHOLDS (Absolute goodness-of-fit)
    # -------------------------------------------------------------------------
    ppc_choice_mae_max: float = 0.05
    ppc_choice_max_err: float = 0.10
    ppc_rt_quantile_mae_max: float = 0.05
    ppc_rt_quantile_max_err: float = 0.10

    # -------------------------------------------------------------------------
    # 7. MODEL ARCHITECTURES & DYNAMIC PROTOCOLS
    # -------------------------------------------------------------------------
    final_core_models: List[str] = field(
        default_factory=lambda: ['null', 'v', 'a', 'va']
    )
    final_exploratory_models: List[str] = field(
        default_factory=lambda: ['vaz', 'vazt']
    )

    # Dynamic MCMC protocols (computed in __post_init__)
    mcmc_protocols: Dict[str, Dict[str, Any]] = field(init=False)

    # -------------------------------------------------------------------------
    # DYNAMIC INITIALIZATION
    # -------------------------------------------------------------------------
    def _infer_model_tier(self, model_name: str) -> str:
        """Infers complexity tier based on free parameter families."""
        name = model_name.lower()
        free_families = sum([
            'v' in name, 'a' in name, 'z' in name, 't' in name
        ])
        has_hard_params = ('z' in name) or ('t' in name)

        if free_families >= 4 or (free_families >= 3 and has_hard_params):
            return "complex"
        elif free_families >= 3:
            return "medium"
        else:
            return "simple"

    def __post_init__(self) -> None:
        """Calculates total required MCMC iterations per model tier."""
        self.mcmc_protocols = {}
        all_models = self.final_core_models + self.final_exploratory_models

        # Tier-specific sampling constraints
        tiers_config = {
            "simple":  {"burn": 2000, "target_kept": 3000, "max_kept": 8000},
            "medium":  {"burn": 3000, "target_kept": 4000, "max_kept": 10000},
            "complex": {"burn": 4000, "target_kept": 5000, "max_kept": 12000},
        }

        for model_name in all_models:
            tier = self._infer_model_tier(model_name)
            proto = tiers_config[tier]

            if self.run_mode == 'debug':
                self.mcmc_protocols[model_name] = {
                    "tier": tier,
                    "n_samples": 600,    # total samples = burn + kept
                    "burn": 100,
                    "thin": 1
                }
            else:
                # n_samples = burn + target_kept (dockerHDDM subtracts burn)
                self.mcmc_protocols[model_name] = {
                    "tier": tier,
                    "n_samples": proto["burn"] + proto["target_kept"],
                    "burn": proto["burn"],
                    "thin": self.default_thin,
                    "max_samples": proto["burn"] + proto["max_kept"]
                }

    # -------------------------------------------------------------------------
    # PARAMETER CLASSIFICATION
    # -------------------------------------------------------------------------
    @staticmethod
    def identify_focal_parameters(param_names: List[str]) -> List[str]:
        """
        Separates focal inferential parameters (group-level Intercepts and
        Treatment contrasts) from hierarchical nuisance parameters
        (subject-level deviations, transformed scales).
        """
        nuisance_pattern = re.compile(r"(_subj|_trans|_std|\.\d+$)")
        return [p for p in param_names if not nuisance_pattern.search(p)]

    # -------------------------------------------------------------------------
    # COMPUTED PROPERTIES
    # -------------------------------------------------------------------------
    @property
    def final_all_models(self) -> List[str]:
        """Aggregation of core and exploratory model architectures."""
        return self.final_core_models + self.final_exploratory_models

    # -------------------------------------------------------------------------
    # DIRECTORY MANAGEMENT
    # -------------------------------------------------------------------------
    def initialize_directories(self) -> Dict[str, Path]:
        """Constructs and validates the publication output directory tree."""
        subdirs = [
            'manifests', 'audit', 'models', 'ppc', 'recovery',
            'figures_main', 'figures_supp', 'tables_main', 'tables_supp'
        ]
        root_out = self.base_dir / f"results_hddm_{self.run_mode}"
        paths = {name: root_out / name for name in subdirs}
        for p in paths.values():
            p.mkdir(parents=True, exist_ok=True)
        return paths

    # -------------------------------------------------------------------------
    # CRYPTOGRAPHIC FINGERPRINT
    # -------------------------------------------------------------------------
    def generate_cryptographic_fingerprint(self) -> Dict[str, Any]:
        """
        Computes deterministic SHA-256 hash of structural hyperparameters.
        Aesthetic parameters (colors, labels) are explicitly excluded.
        """
        critical_keys = [
            'run_mode', 'include_params',
            'group_only_regressors', 'p_outlier', 'use_informative_priors',
            'baseline_condition', 'n_chains', 'base_seed',
            'rhat_focal', 'ess_bulk_focal', 'ess_tail_focal',
            'rhat_nuisance', 'ess_bulk_nuisance', 'ess_tail_nuisance',
            'default_thin', 'enable_loglike', 'enable_ppc',
            'ppc_choice_mae_max', 'ppc_rt_quantile_mae_max',
            'final_core_models', 'final_exploratory_models',
            'mcmc_protocols'
        ]

        cfg_dict = asdict(self)
        structural_params = {k: cfg_dict[k] for k in critical_keys}

        # Deterministic serialization requires sorted keys
        json_str = json.dumps(structural_params, sort_keys=True)
        config_hash = hashlib.sha256(json_str.encode('utf-8')).hexdigest()

        return {
            'timestamp_utc': datetime.utcnow().isoformat() + 'Z',
            'config_hash': config_hash,
            'structural_parameters': structural_params
        }


# =============================================================================
# EXECUTION & PERSISTENCE
# =============================================================================

# Instantiate configuration in active kernel memory
CFG = HDDMConfig()
PATHS = CFG.initialize_directories()


def _serialize_config_to_disk(
    cfg_obj: HDDMConfig, target_paths: Dict[str, Path]
) -> str:
    """
    Serializes the configuration state to a Python module ('hddm_config.py')
    for cross-session recovery after kernel restarts. Also injects the
    utility functions (validate_pipeline_lineage, identify_winning_model)
    and class methods into the generated module.
    """
    py_config_path = 'hddm_config.py'
    with open(py_config_path, 'w', encoding='utf-8') as f:
        f.write("# -*- coding: utf-8 -*-\n")
        f.write("# Auto-generated SSOT configuration module\n")
        f.write("import re\nimport json\nimport pandas as pd\n")
        f.write("from pathlib import Path\n\n")

        # --- Write utility functions ---
        f.write("def validate_pipeline_lineage(paths, logger=None):\n")
        f.write("    fp = paths['manifests'] / 'config_fingerprint.json'\n")
        f.write("    if not fp.exists():\n")
        f.write("        raise FileNotFoundError('Missing fingerprint.')\n")
        f.write("    with open(fp, 'r') as fh:\n")
        f.write("        h = json.load(fh).get('config_hash', '')\n")
        f.write("    if logger: logger.info(f'Lineage: {h[:16]}...')\n")
        f.write("    return h\n\n")

        f.write("def identify_winning_model(paths):\n")
        f.write("    for p in [paths['audit']/'final_model_selection_audit.csv',\n")
        f.write("              paths['tables_main']/'model_comparison_summary.csv']:\n")
        f.write("        if p.exists():\n")
        f.write("            df = pd.read_csv(p)\n")
        f.write("            if 'model_name' in df.columns:\n")
        f.write("                return str(df['model_name'].iloc[0]).lower()\n")
        f.write("            if 'Model' in df.columns and 'Is_Winner' in df.columns:\n")
        f.write("                mask = df['Is_Winner'].astype(str).str.strip().str.lower()=='true'\n")
        f.write("                w = df[mask]\n")
        f.write("                if not w.empty: return str(w.iloc[0]['Model']).lower()\n")
        f.write("    raise FileNotFoundError('No audit manifest found.')\n\n")

        # --- Write config class ---
        f.write("class _HDDMConfig:\n")

        # Static attributes
        for key, value in cfg_obj.__dict__.items():
            if isinstance(value, Path):
                f.write(f"    {key} = Path(r'{value}')\n")
            elif isinstance(value, str):
                f.write(f"    {key} = '{value}'\n")
            else:
                f.write(f"    {key} = {repr(value)}\n")

        # Inject properties and methods
        f.write("\n    @property\n    def final_all_models(self):\n")
        f.write("        return self.final_core_models + self.final_exploratory_models\n")

        f.write("\n    def initialize_directories(self):\n")
        f.write("        subdirs = ['manifests','audit','models','ppc','recovery',")
        f.write("'figures_main','figures_supp','tables_main','tables_supp']\n")
        f.write("        root = self.base_dir / f'results_hddm_{self.run_mode}'\n")
        f.write("        paths = {n: root / n for n in subdirs}\n")
        f.write("        for p in paths.values(): p.mkdir(parents=True, exist_ok=True)\n")
        f.write("        return paths\n")

        f.write("\n    @staticmethod\n")
        f.write("    def identify_focal_parameters(param_names):\n")
        f.write("        import re\n")
        f.write("        pat = re.compile(r'(_subj|_trans|_std|\\.\\d+$)')\n")
        f.write("        return [p for p in param_names if not pat.search(p)]\n")

        # Instantiate at module level
        f.write("\nCFG = _HDDMConfig()\n")

    # --- Persist cryptographic fingerprint ---
    fingerprint = cfg_obj.generate_cryptographic_fingerprint()
    fp_path = target_paths['manifests'] / "config_fingerprint.json"
    with open(fp_path, 'w', encoding='utf-8') as f:
        json.dump(fingerprint, f, indent=2, default=str)

    return fingerprint['config_hash']


# Execute serialization
config_hash = _serialize_config_to_disk(CFG, PATHS)

# Print confirmation
print("=" * 70)
print(f"  STEP 2a: Configuration SSOT Initialized")
print(f"  Run Mode:         {CFG.run_mode}")
print(f"  DDM Parameters:   v, a, t + {CFG.include_params}")
print(f"  Chains:           {CFG.n_chains}")
print(f"  LogLike (WAIC):   {CFG.enable_loglike}")
print(f"  PPC Enabled:      {CFG.enable_ppc}")
print(f"  Models:           {len(CFG.final_all_models)} architectures")
print(f"  Config Hash:      {config_hash[:24]}...")
print("=" * 70)

# Display per-model protocols
for model_name in CFG.final_all_models:
    proto = CFG.mcmc_protocols[model_name]
    print(
        f"  {model_name:>6s} | tier={proto['tier']:>7s} | "
        f"samples={proto['n_samples']:>5d} | burn={proto['burn']:>4d}"
    )

# Step 2b: Hierarchical Bayesian Model Specification and MCMC Estimation

In [ ]:
# -*- coding: utf-8 -*-
"""
=============================================================================
STEP 2b: Hierarchical Bayesian Model Specification & Adaptive MCMC
         Estimation (dockerHDDM ArviZ-Centric Workflow)
=============================================================================
Description:
  - Leverages dockerHDDM's native parallel chain execution and automatic
    InferenceData construction (Pan et al., 2025), eliminating the need
    for manual multiprocessing and ArviZ tensor assembly.
  - Implements a Dynamic Regressor Factory for automated Patsy formula
    generation from model architecture strings (e.g., 'vazt' -> four
    separate Treatment-coded regressors).
  - Validates empirical data integrity before committing to MCMC estimation.
  - Executes an adaptive convergence loop: after initial sampling, evaluates
    stratified diagnostics (focal vs nuisance parameters) via ArviZ, and
    appends additional samples if thresholds are not met.
  - Handles memory-constrained environments gracefully: if loglike=True
    triggers an OutOfMemoryError, the pipeline automatically retries
    without pointwise log-likelihood computation.
  - Exports a structured model manifest CSV for downstream consumption
    by Steps 3-6, stamped with the active configuration hash.
=============================================================================
"""

import os
import gc
import time
import json
import logging
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Any

import numpy as np
import pandas as pd
import arviz as az
import hddm

# -----------------------------------------------------------------------------
# CONFIGURATION IMPORT & ENVIRONMENT SETUP
# -----------------------------------------------------------------------------
try:
    from hddm_config import CFG, validate_pipeline_lineage
except ImportError:
    raise ImportError(
        "CRITICAL ERROR: 'hddm_config.py' not found. "
        "Execute Step 2a to generate the global configuration SSOT."
    )

PATHS = CFG.initialize_directories()


# =============================================================================
# LOGGING SETUP (Disk + Console, avoids basicConfig collision in notebooks)
# =============================================================================
def _setup_estimation_logger() -> logging.Logger:
    """Initializes dual-sink logging for the MCMC estimation sequence."""
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    logger = logging.getLogger(f'hddm_mcmc_{ts}')
    logger.handlers = []
    logger.setLevel(logging.INFO)

    fmt = logging.Formatter(
        '%(asctime)s - %(levelname)s - %(message)s', datefmt='%H:%M:%S'
    )
    fh = logging.FileHandler(
        PATHS['audit'] / f'mcmc_estimation_{ts}.log', encoding='utf-8'
    )
    ch = logging.StreamHandler()
    fh.setFormatter(fmt)
    ch.setFormatter(fmt)

    logger.addHandler(fh)
    logger.addHandler(ch)
    return logger


# =============================================================================
# DYNAMIC REGRESSOR FACTORY
# =============================================================================
def build_model_regressors(model_name: str, baseline: str) -> List[str]:
    """
    Dynamically constructs Patsy Treatment-coded formulas for HDDMRegressor
    based on which DDM parameter families are freely varying in the model.

    For 'null' models, returns an empty list (no regressors needed).
    For named models like 'va', returns regressors for v and a.

    Parameters:
      model_name: Architecture string (e.g., 'null', 'v', 'va', 'vazt').
      baseline:   Reference level for Treatment coding (e.g., 'neu').

    Returns:
      List of Patsy formula strings for HDDMRegressor, or empty list.
    """
    regressors = []
    name_lower = model_name.lower()

    if name_lower == 'null':
        return regressors

    for param in ['v', 'a', 'z', 't']:
        if param in name_lower:
            regressors.append(
                f"{param} ~ C(emotion, Treatment('{baseline}'))"
            )

    return regressors


# =============================================================================
# DATA VALIDATION
# =============================================================================
def validate_empirical_data(
    data_path: str, logger: logging.Logger
) -> pd.DataFrame:
    """
    Loads and enforces datatype constraints on the empirical design matrix.
    Verifies that mandatory HDDM columns ('subj_idx', 'rt', 'response')
    are present, that the baseline condition exists in the data, and that
    rt values are positive and in seconds.
    """
    if not os.path.exists(data_path):
        raise FileNotFoundError(f"Input data not found: {data_path}")

    df = pd.read_csv(data_path)
    logger.info(f"Loaded empirical data: {df.shape[0]} trials, {df.shape[1]} columns")

    # Mandatory column check (HDDM requirement)
    required_cols = {'subj_idx', 'rt', 'response'}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing mandatory HDDM columns: {missing}")

    # Ensure correct dtypes
    df['subj_idx'] = df['subj_idx'].astype(str)
    df['rt'] = pd.to_numeric(df['rt'], errors='coerce')
    df['response'] = pd.to_numeric(df['response'], errors='coerce')

    # Drop rows with invalid rt or response
    n_before = len(df)
    df = df.dropna(subset=['rt', 'response'])
    n_dropped = n_before - len(df)
    if n_dropped > 0:
        logger.warning(f"Dropped {n_dropped} rows with invalid rt/response values.")

    # Verify baseline condition exists
    if 'emotion' in df.columns:
        if CFG.baseline_condition not in df['emotion'].unique():
            raise ValueError(
                f"Baseline condition '{CFG.baseline_condition}' not found "
                f"in data. Available: {df['emotion'].unique().tolist()}"
            )

    # Summary statistics
    n_subjects = df['subj_idx'].nunique()
    n_conditions = df['emotion'].nunique() if 'emotion' in df.columns else 0
    logger.info(
        f"Validated: {n_subjects} subjects, {n_conditions} conditions, "
        f"{len(df)} clean trials"
    )
    logger.info(
        f"RT range: [{df['rt'].min():.3f}, {df['rt'].max():.3f}] seconds"
    )

    return df


# =============================================================================
# STRATIFIED CONVERGENCE DIAGNOSTICS
# =============================================================================
def evaluate_stratified_convergence(
    infdata: az.InferenceData, logger: logging.Logger
) -> Tuple[bool, Dict[str, float], pd.DataFrame]:
    """
    Evaluates MCMC convergence using stratified criteria from CFG:
    focal parameters (group-level effects) require stricter thresholds
    than nuisance parameters (subject-level deviations).

    This function operates directly on the InferenceData object returned
    by dockerHDDM's model.sample(return_infdata=True), leveraging ArviZ's
    native summary computation which correctly handles multi-chain geometry.

    Returns:
      converged:  Boolean indicating whether all criteria are met.
      metrics:    Dictionary of diagnostic values for logging.
      summary_df: Full ArviZ summary DataFrame with param_class column.
    """
    summary_df = az.summary(infdata, round_to=4, hdi_prob=0.95)

    # Classify parameters into focal vs nuisance
    focal_params = CFG.identify_focal_parameters(summary_df.index.tolist())
    summary_df['param_class'] = [
        'focal' if p in focal_params else 'nuisance'
        for p in summary_df.index
    ]

    focal_df = summary_df[summary_df["param_class"] == "focal"]
    nuisance_df = summary_df[summary_df["param_class"] == "nuisance"]

    # Compute diagnostic metrics
    metrics = {
        "f_rhat": focal_df["r_hat"].max() if not focal_df.empty else 1.0,
        "f_bulk": focal_df["ess_bulk"].min() if not focal_df.empty else float("inf"),
        "f_tail": (
            focal_df["ess_tail"].min()
            if ("ess_tail" in focal_df.columns and not focal_df.empty)
            else float("inf")
        ),
        "n_rhat": nuisance_df["r_hat"].max() if not nuisance_df.empty else 1.0,
        "n_bulk": nuisance_df["ess_bulk"].min() if not nuisance_df.empty else float("inf"),
        "n_tail": (
            nuisance_df["ess_tail"].min()
            if ("ess_tail" in nuisance_df.columns and not nuisance_df.empty)
            else float("inf")
        ),
    }

    # Evaluate focal criteria
    focal_ok = (
        (metrics["f_rhat"] <= CFG.rhat_focal) and
        (metrics["f_bulk"] >= CFG.ess_bulk_focal) and
        (metrics["f_tail"] >= CFG.ess_tail_focal)
    )

    # Evaluate nuisance criteria
    nuisance_ok = (
        (metrics["n_rhat"] <= CFG.rhat_nuisance) and
        (metrics["n_bulk"] >= CFG.ess_bulk_nuisance) and
        (metrics["n_tail"] >= CFG.ess_tail_nuisance)
    )

    converged = focal_ok and nuisance_ok

    # Log detailed diagnostics
    logger.info(
        f"  FOCAL   | R-hat={metrics['f_rhat']:.3f} "
        f"(≤{CFG.rhat_focal}) | "
        f"ESS_bulk={metrics['f_bulk']:.0f} "
        f"(≥{CFG.ess_bulk_focal}) | "
        f"ESS_tail={metrics['f_tail']:.0f} "
        f"(≥{CFG.ess_tail_focal}) | "
        f"{'PASS' if focal_ok else 'FAIL'}"
    )
    logger.info(
        f"  NUISANCE| R-hat={metrics['n_rhat']:.3f} "
        f"(≤{CFG.rhat_nuisance}) | "
        f"ESS_bulk={metrics['n_bulk']:.0f} "
        f"(≥{CFG.ess_bulk_nuisance}) | "
        f"ESS_tail={metrics['n_tail']:.0f} "
        f"(≥{CFG.ess_tail_nuisance}) | "
        f"{'PASS' if nuisance_ok else 'FAIL'}"
    )

    return converged, metrics, summary_df


# =============================================================================
# CORE: SINGLE MODEL FITTING PIPELINE
# =============================================================================
def fit_single_model(
    model_name: str,
    data: pd.DataFrame,
    logger: logging.Logger,
    config_hash: str
) -> Dict[str, Any]:
    """
    Fits a single HDDM model architecture using dockerHDDM's native
    ArviZ-centric sampling interface. Implements:
      1. Dynamic model construction (HDDM vs HDDMRegressor).
      2. Initial MCMC sampling with parallel chains.
      3. Adaptive convergence loop with stratified diagnostics.
      4. Memory-safe loglike computation with automatic fallback.
      5. Model and InferenceData serialization to disk.

    Parameters:
      model_name:  Architecture string (e.g., 'null', 'v', 'va', 'vazt').
      data:        Validated empirical DataFrame.
      logger:      Configured logging instance.
      config_hash: Active SHA-256 fingerprint for lineage tracking.

    Returns:
      Dictionary containing model metadata, convergence status, file paths,
      and diagnostic metrics for the model manifest.
    """
    logger.info(f"\n{'='*70}")
    logger.info(f"FITTING MODEL: [{model_name.upper()}]")
    logger.info(f"{'='*70}")

    proto = CFG.mcmc_protocols[model_name]
    logger.info(
        f"  Protocol: tier={proto['tier']}, "
        f"n_samples={proto['n_samples']}, burn={proto['burn']}, "
        f"chains={CFG.n_chains}"
    )

    # -----------------------------------------------------------------
    # PHASE 1: Model Construction
    # -----------------------------------------------------------------
    regressors = build_model_regressors(model_name, CFG.baseline_condition)

    if not regressors:
        # Null model or intercept-only: use base HDDM
        model = hddm.HDDM(
            data,
            include=CFG.include_params,
            is_group_model=True,
            informative=CFG.use_informative_priors,
            p_outlier=CFG.p_outlier
        )
        logger.info(f"  Constructed: hddm.HDDM (intercept-only)")
    else:
        # Regression model with Treatment-coded condition effects
        model = hddm.HDDMRegressor(
            data,
            regressors,
            include=CFG.include_params,
            group_only_regressors=CFG.group_only_regressors,
            keep_regressor_trace=CFG.keep_regressor_trace,
            informative=CFG.use_informative_priors,
            p_outlier=CFG.p_outlier
        )
        logger.info(f"  Constructed: hddm.HDDMRegressor")
        for reg in regressors:
            logger.info(f"    -> {reg}")

    # -----------------------------------------------------------------
    # PHASE 2: Initial MCMC Sampling (dockerHDDM native interface)
    # -----------------------------------------------------------------
    save_prefix = str(PATHS['models'] / f"hddm_{model_name}")
    enable_loglike = CFG.enable_loglike

    t_start = time.time()
    logger.info(
        f"  Sampling: {proto['n_samples']} iterations × "
        f"{CFG.n_chains} chains (burn={proto['burn']})..."
    )

    try:
        # Primary attempt: full ArviZ-centric sampling with loglike
        infdata = model.sample(
            proto['n_samples'],
            burn=proto['burn'],
            thin=proto.get('thin', 1),
            chains=CFG.n_chains,
            return_infdata=True,
            loglike=enable_loglike,
            ppc=CFG.enable_ppc,
            n_ppc=CFG.ppc_samples if CFG.enable_ppc else None,
            save_name=save_prefix
        )
    except MemoryError:
        # Memory-safe fallback: retry without pointwise log-likelihood
        logger.warning(
            "  MemoryError encountered with loglike=True. "
            "Retrying without log-likelihood (DIC fallback)..."
        )
        enable_loglike = False
        gc.collect()

        infdata = model.sample(
            proto['n_samples'],
            burn=proto['burn'],
            thin=proto.get('thin', 1),
            chains=CFG.n_chains,
            return_infdata=True,
            loglike=False,
            ppc=CFG.enable_ppc,
            n_ppc=CFG.ppc_samples if CFG.enable_ppc else None,
            save_name=save_prefix
        )

    t_elapsed = time.time() - t_start
    logger.info(f"  Initial sampling completed in {t_elapsed/60:.1f} min")

    # -----------------------------------------------------------------
    # PHASE 3: Adaptive Convergence Loop
    # -----------------------------------------------------------------
    converged, metrics, summary_df = evaluate_stratified_convergence(
        infdata, logger
    )

    total_samples = proto['n_samples']
    max_samples = proto.get('max_samples', proto['n_samples'] * 2)
    cycle = 0

    while not converged and cycle < CFG.max_adaptive_cycles:
        cycle += 1

        # Calculate additional samples needed based on ESS deficit ratio
        target_ess = CFG.ess_bulk_focal
        current_min_ess = max(metrics["f_bulk"], 1.0)
        deficit_ratio = target_ess / current_min_ess
        added_samples = max(1000, int(total_samples * (deficit_ratio - 1) * 1.25))

        # Enforce ceiling
        if total_samples + added_samples > max_samples:
            added_samples = max_samples - total_samples
            if added_samples <= 0:
                logger.warning(
                    f"  Max sample ceiling ({max_samples}) reached. "
                    f"Stopping adaptive extension."
                )
                break

        logger.info(
            f"\n  [Adaptive Cycle {cycle}/{CFG.max_adaptive_cycles}] "
            f"Adding {added_samples} samples (ESS deficit ratio: {deficit_ratio:.2f})..."
        )

        # dockerHDDM continuation: sample additional iterations
        # Note: burn=0 for continuation, new samples are appended
        model.sample(
            added_samples,
            burn=0,
            thin=proto.get('thin', 1),
            chains=CFG.n_chains,
            return_infdata=True,
            loglike=False,       # Skip loglike during adaptive phase
            ppc=False,           # Skip PPC during adaptive phase
            save_name=save_prefix
        )

        total_samples += added_samples

        # Re-evaluate with updated InferenceData
        # Reload from disk to get the updated state
        infdata = az.from_netcdf(f"{save_prefix}.nc")
        converged, metrics, summary_df = evaluate_stratified_convergence(
            infdata, logger
        )

    # -----------------------------------------------------------------
    # PHASE 4: Export Diagnostics and Cleanup
    # -----------------------------------------------------------------
    # Save full summary to audit directory
    summary_path = PATHS['audit'] / f"summary_{model_name}.csv"
    summary_df.to_csv(summary_path)
    logger.info(f"  Summary exported: {summary_path.name}")

    # Extract DIC (always available from the HDDM model object)
    try:
        dic_value = model.dic
    except Exception:
        dic_value = float('inf')
    logger.info(f"  DIC = {dic_value:.2f}")

    # Construct manifest record
    manifest_record = {
        'config_hash': config_hash,
        'model_name': model_name,
        'tier': proto['tier'],
        'n_chains': CFG.n_chains,
        'total_samples': total_samples,
        'burn': proto['burn'],
        'converged': converged,
        'dic': dic_value,
        'loglike_available': enable_loglike,
        'ppc_available': CFG.enable_ppc,
        'f_rhat_max': metrics.get('f_rhat', np.nan),
        'f_ess_bulk_min': metrics.get('f_bulk', np.nan),
        'f_ess_tail_min': metrics.get('f_tail', np.nan),
        'n_rhat_max': metrics.get('n_rhat', np.nan),
        'n_ess_bulk_min': metrics.get('n_bulk', np.nan),
        'elapsed_minutes': t_elapsed / 60,
        'nc_path': f"{save_prefix}.nc",
        'hddm_path': f"{save_prefix}.hddm"
    }

    logger.info(
        f"  STATUS: {'CONVERGED' if converged else 'NOT CONVERGED'} "
        f"after {total_samples} total samples"
    )

    # Memory cleanup
    del model
    gc.collect()

    return manifest_record


# =============================================================================
# MASTER PIPELINE EXECUTION
# =============================================================================
def run_estimation_pipeline():
    """
    Orchestrates the sequential estimation of all model architectures
    defined in CFG. Validates data once, then iterates through each model,
    collecting manifest records for downstream Steps.
    """
    logger = _setup_estimation_logger()

    # Validate lineage
    config_hash = validate_pipeline_lineage(PATHS, logger)

    logger.info("=" * 70)
    logger.info(
        f"HDDM DOCKERHDDM ESTIMATION PIPELINE INITIATED "
        f"(Mode: {CFG.run_mode.upper()})"
    )
    logger.info(f"Active Lineage Hash: {config_hash[:24]}...")
    logger.info(f"DDM Parameters: v, a, t + {CFG.include_params}")
    logger.info(f"Parallel Chains: {CFG.n_chains}")
    logger.info(
        f"ArviZ Features: loglike={CFG.enable_loglike}, "
        f"ppc={CFG.enable_ppc}"
    )
    logger.info(f"Models to fit: {CFG.final_all_models}")
    logger.info("=" * 70)

    if CFG.run_mode == 'debug':
        logger.warning(
            "!!! RUNNING IN DEBUG MODE: Results are NOT publication-grade. !!!"
        )
        time.sleep(2)

    # Phase 1: Validate empirical data (once for all models)
    data_path = CFG.base_dir / "hddm_data_unfair.csv"
    data = validate_empirical_data(str(data_path), logger)

    # Phase 2: Iterate over all model architectures
    manifest_records = []

    for model_name in CFG.final_all_models:
        try:
            record = fit_single_model(model_name, data, logger, config_hash)
            manifest_records.append(record)
        except Exception as e:
            logger.error(
                f"CRITICAL: Model [{model_name}] failed: {e}",
                exc_info=True
            )
            manifest_records.append({
                'config_hash': config_hash,
                'model_name': model_name,
                'tier': CFG.mcmc_protocols[model_name]['tier'],
                'converged': False,
                'dic': float('inf'),
                'error': str(e)
            })

    # Phase 3: Export model manifest
    df_manifest = pd.DataFrame(manifest_records)
    manifest_path = PATHS['manifests'] / "model_manifest.csv"
    df_manifest.to_csv(manifest_path, index_col=False) # explicitly avoid writing index to match standard IO
    logger.info(f"\nModel manifest exported: {manifest_path.name}")

    # Summary report
    n_converged = df_manifest['converged'].sum() if 'converged' in df_manifest else 0
    logger.info(f"\n{'='*70}")
    logger.info(
        f"ESTIMATION COMPLETE: {n_converged}/{len(manifest_records)} "
        f"models converged"
    )
    logger.info(f"{'='*70}")

    # Display summary table
    display_cols = [
        'model_name', 'tier', 'converged', 'dic',
        'f_rhat_max', 'f_ess_bulk_min', 'elapsed_minutes'
    ]
    available_cols = [c for c in display_cols if c in df_manifest.columns]
    print("\n--- Estimation Summary ---")
    print(df_manifest[available_cols].to_string(index=False))


# =============================================================================
# ENTRY POINT
# =============================================================================
if __name__ == "__main__":
    run_estimation_pipeline()

# Step 3: Posterior Trace Extraction and Predictive Simulation

In [ ]:
# -*- coding: utf-8 -*-
"""
=============================================================================
STEP 3: Posterior Predictive Checks (PPC) & Diagnostic Visualization
=============================================================================
Description:
  - Consumes InferenceData artifacts (.nc files) produced by Step 2b's
    dockerHDDM native sampling (return_infdata=True, ppc=True).
  - Executes rigorous PPC adequacy evaluation at two levels:
    1. Condition-level: Tests whether the model reproduces empirical
       rejection rates per emotion condition (choice MAE).
    2. RT-quantile-level: Tests whether the model reproduces the shape
       of empirical RT distributions (10th/50th/90th percentile MAE).
  - Generates publication-ready diagnostic visualizations using ArviZ:
    * az.plot_ppc() for individual-level and condition-level density
      comparisons (Pan et al., 2025, Fig. 6).
    * az.plot_trace() for MCMC chain mixing inspection.
  - Exports structured PPC summary statistics stamped with the active
    configuration hash for consumption by Step 4's diagnostic funnel.
  - Validates cryptographic lineage and verifies manifest consistency
    before processing any model artifacts.
=============================================================================
"""

import os
import gc
import json
import logging
import warnings
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt
import seaborn as sns
import hddm

warnings.filterwarnings('ignore', category=FutureWarning)

# -----------------------------------------------------------------------------
# CONFIGURATION IMPORT & ENVIRONMENT SETUP
# -----------------------------------------------------------------------------
try:
    from hddm_config import CFG, validate_pipeline_lineage
except ImportError:
    raise ImportError(
        "CRITICAL ERROR: 'hddm_config.py' not found. "
        "Execute Step 2a to generate the global configuration SSOT."
    )

PATHS = CFG.initialize_directories()

# Apply academic visualization aesthetics
sns.set_theme(style="ticks")
plt.rcParams.update({
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'legend.frameon': False
})


# =============================================================================
# LOGGING SETUP
# =============================================================================
def _setup_ppc_logger() -> logging.Logger:
    """Initializes dual-sink logging for the PPC audit process."""
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    logger = logging.getLogger(f'hddm_ppc_{ts}')
    logger.handlers = []
    logger.setLevel(logging.INFO)

    fmt = logging.Formatter(
        '%(asctime)s - %(levelname)s - %(message)s', datefmt='%H:%M:%S'
    )
    fh = logging.FileHandler(
        PATHS['audit'] / f'ppc_audit_{ts}.log', encoding='utf-8'
    )
    ch = logging.StreamHandler()
    fh.setFormatter(fmt)
    ch.setFormatter(fmt)

    logger.addHandler(fh)
    logger.addHandler(ch)
    return logger


# =============================================================================
# CORE CLASS: PPC AUDIT & DIAGNOSTIC ENGINE
# =============================================================================
class PPCAuditEngine:
    """
    Loads InferenceData from Step 2b, executes multi-level PPC adequacy
    checks, generates ArviZ-based diagnostic visualizations, and exports
    structured summaries for the Step 4 diagnostic funnel.
    """

    def __init__(self):
        self.logger = _setup_ppc_logger()
        self.active_hash = validate_pipeline_lineage(PATHS, self.logger)
        self.manifest = self._load_and_validate_manifest()

        self.observed_stats: List[Dict] = []
        self.ppc_stats: List[Dict] = []

        self.logger.info("=" * 70)
        self.logger.info(
            f"PPC AUDIT ENGINE INITIATED (Mode: {CFG.run_mode.upper()})"
        )
        self.logger.info(f"Active Lineage Hash: {self.active_hash[:24]}...")
        self.logger.info(
            f"Models to audit: {len(self.manifest)} architectures"
        )
        self.logger.info("=" * 70)

    def _load_and_validate_manifest(self) -> pd.DataFrame:
        """
        Reads the model manifest produced by Step 2b. Verifies that the
        configuration hash matches the active session to prevent stale
        artifact consumption.
        """
        manifest_path = PATHS['manifests'] / "model_manifest.csv"
        if not manifest_path.exists():
            raise FileNotFoundError(
                "Model manifest not found. Execute Step 2b first."
            )

        df = pd.read_csv(manifest_path)
        if df.empty:
            raise RuntimeError("Model manifest is empty. Step 2b likely failed.")

        # Cross-validate lineage hash
        if 'config_hash' in df.columns:
            artifact_hash = str(df['config_hash'].iloc[0])
            if self.active_hash != artifact_hash:
                raise RuntimeError(
                    f"\nCRITICAL LINEAGE MISMATCH!\n"
                    f"Current Hash: {self.active_hash}\n"
                    f"Manifest Hash: {artifact_hash}\n"
                    f"Re-execute Step 2b to align with current configuration."
                )

        return df

    # -----------------------------------------------------------------
    # ARVIZ-BASED DIAGNOSTIC VISUALIZATION
    # -----------------------------------------------------------------
    def _generate_trace_plots(
        self, infdata: az.InferenceData, model_name: str
    ):
        """
        Generates MCMC trace plots (posterior density + chain traces)
        for focal parameters. Essential for visual convergence inspection.
        Ref: Pan et al. (2025), Fig. 5a.
        """
        # Identify focal parameters present in the InferenceData
        all_vars = list(infdata.posterior.data_vars.keys())
        focal_vars = CFG.identify_focal_parameters(all_vars)

        if not focal_vars:
            self.logger.warning(
                f"  [{model_name}] No focal parameters found for trace plots."
            )
            return

        try:
            az.plot_trace(
                infdata,
                var_names=focal_vars,
                compact=True,
                figsize=(12, 2.5 * len(focal_vars))
            )
            plt.tight_layout()
            out_path = (
                PATHS['figures_supp'] /
                f"diagnostics_trace_{model_name}.pdf"
            )
            plt.savefig(out_path, dpi=300, bbox_inches='tight')
            plt.close()
            self.logger.info(f"  Trace plot saved: {out_path.name}")
        except Exception as e:
            self.logger.warning(f"  Trace plot failed for [{model_name}]: {e}")

    def _generate_ppc_density_plots(
        self, infdata: az.InferenceData, model_name: str
    ):
        """
        Generates posterior predictive density comparison plots using
        az.plot_ppc(). Shows observed RT distribution overlaid with
        posterior predictive samples.
        Ref: Pan et al. (2025), Fig. 6.
        """
        if not hasattr(infdata, 'posterior_predictive'):
            self.logger.warning(
                f"  [{model_name}] No posterior_predictive group in "
                f"InferenceData. PPC plots skipped."
            )
            return

        try:
            # Global PPC density plot
            ax = az.plot_ppc(
                infdata,
                var_names=['rt'],
                num_pp_samples=100,
                flatten=[]
            )
            plt.tight_layout()
            out_path = (
                PATHS['figures_supp'] /
                f"ppc_density_global_{model_name}.pdf"
            )
            plt.savefig(out_path, dpi=300, bbox_inches='tight')
            plt.close()
            self.logger.info(f"  PPC density plot saved: {out_path.name}")
        except Exception as e:
            self.logger.warning(
                f"  PPC density plot failed for [{model_name}]: {e}"
            )

    def _generate_rank_plots(
        self, infdata: az.InferenceData, model_name: str
    ):
        """
        Generates rank plots (ArviZ) for focal parameters.
        Rank plots are a more informative alternative to traditional
        trace plots for diagnosing chain mixing issues.
        """
        all_vars = list(infdata.posterior.data_vars.keys())
        focal_vars = CFG.identify_focal_parameters(all_vars)

        if not focal_vars:
            return

        try:
            az.plot_rank(
                infdata,
                var_names=focal_vars,
                kind='vlines',
                vlines_kwargs={'lw': 0},
                marker_vlines_kwargs={'lw': 2}
            )
            plt.tight_layout()
            out_path = (
                PATHS['figures_supp'] /
                f"diagnostics_rank_{model_name}.pdf"
            )
            plt.savefig(out_path, dpi=300, bbox_inches='tight')
            plt.close()
            self.logger.info(f"  Rank plot saved: {out_path.name}")
        except Exception as e:
            self.logger.warning(f"  Rank plot failed for [{model_name}]: {e}")

    # -----------------------------------------------------------------
    # QUANTITATIVE PPC ADEQUACY EVALUATION
    # -----------------------------------------------------------------
    def _compute_ppc_adequacy_from_hddm(
        self, model_name: str
    ) -> Dict[str, float]:
        """
        Computes quantitative PPC adequacy metrics by loading the HDDM
        model object and generating posterior predictive data.

        This method is used as a complement to the ArviZ PPC visualization,
        providing numerical adequacy measures (rejection rate MAE, RT
        quantile MAE) that the Step 4 funnel uses for model filtering.

        The HDDM model object is needed here because dockerHDDM's
        InferenceData posterior_predictive group stores predicted RTs
        but the condition-level and response-type disaggregation requires
        access to the original trial structure.
        """
        hddm_path = PATHS['models'] / f"hddm_{model_name}.hddm"
        if not hddm_path.exists():
            self.logger.warning(
                f"  [{model_name}] HDDM model file not found. "
                f"Skipping quantitative PPC."
            )
            return {'choice_mae': np.nan, 'rt_mae': np.nan, 'pass': False}

        model = hddm.load(str(hddm_path))

        # Generate posterior predictive data (condition-aware)
        try:
            ppc_data = hddm.utils.post_pred_gen(
                model, samples=200, append_data=True
            )
        except Exception as e:
            self.logger.warning(
                f"  [{model_name}] post_pred_gen failed: {e}"
            )
            del model
            gc.collect()
            return {'choice_mae': np.nan, 'rt_mae': np.nan, 'pass': False}

        # Compute condition-level observed vs predicted statistics
        obs_records = []
        sim_records = []

        rt_col_sim = 'rt_sampled' if 'rt_sampled' in ppc_data.columns else 'rt'
        resp_col_sim = (
            'response_sampled' if 'response_sampled' in ppc_data.columns
            else 'response'
        )

        if 'emotion' not in ppc_data.columns:
            self.logger.warning(
                f"  [{model_name}] 'emotion' column missing in PPC data."
            )
            del model, ppc_data
            gc.collect()
            return {'choice_mae': np.nan, 'rt_mae': np.nan, 'pass': False}

        for emo in ppc_data['emotion'].unique():
            subset = ppc_data[ppc_data['emotion'] == emo]

            # Observed statistics (from the 'response' and 'rt' columns)
            obs_rej_rate = (subset['response'] == 0).mean()
            obs_records.append({
                'config_hash': self.active_hash,
                'model_name': model_name,
                'emotion': emo,
                'stat_name': 'rejection_rate',
                'stat_value': float(obs_rej_rate),
                'source': 'observed'
            })

            # Simulated statistics
            sim_rej_rate = (subset[resp_col_sim] == 0).mean()
            sim_records.append({
                'config_hash': self.active_hash,
                'model_name': model_name,
                'emotion': emo,
                'stat_name': 'rejection_rate',
                'stat_value': float(sim_rej_rate),
                'source': 'simulated'
            })

            # RT quantiles per response type
            for resp_val, resp_label in [(1, 'accept'), (0, 'reject')]:
                obs_sub = subset[subset['response'] == resp_val]
                sim_sub = subset[subset[resp_col_sim] == resp_val]

                if len(obs_sub) < 5 or len(sim_sub) < 5:
                    continue

                obs_rt = np.abs(obs_sub['rt'].values)
                sim_rt = np.abs(sim_sub[rt_col_sim].values)

                for q_label, q_val in [('rt_q10', 0.10), ('rt_q50', 0.50), ('rt_q90', 0.90)]:
                    obs_records.append({
                        'config_hash': self.active_hash,
                        'model_name': model_name,
                        'emotion': emo,
                        'response_type': resp_label,
                        'stat_name': q_label,
                        'stat_value': float(np.quantile(obs_rt, q_val)),
                        'source': 'observed'
                    })
                    sim_records.append({
                        'config_hash': self.active_hash,
                        'model_name': model_name,
                        'emotion': emo,
                        'response_type': resp_label,
                        'stat_name': q_label,
                        'stat_value': float(np.quantile(sim_rt, q_val)),
                        'source': 'simulated'
                    })

        # Aggregate records
        self.observed_stats.extend(obs_records)
        self.ppc_stats.extend(sim_records)

        # Compute MAE metrics
        df_obs = pd.DataFrame(obs_records)
        df_sim = pd.DataFrame(sim_records)

        if df_obs.empty or df_sim.empty:
            del model, ppc_data
            gc.collect()
            return {'choice_mae': np.nan, 'rt_mae': np.nan, 'pass': False}

        # Merge on shared keys for error computation
        merge_keys = ['emotion', 'stat_name']
        if 'response_type' in df_obs.columns and 'response_type' in df_sim.columns:
            merge_keys.append('response_type')

        merged = pd.merge(
            df_obs[merge_keys + ['stat_value']],
            df_sim[merge_keys + ['stat_value']],
            on=merge_keys,
            suffixes=('_obs', '_sim'),
            how='inner'
        )
        merged['abs_error'] = np.abs(merged['stat_value_obs'] - merged['stat_value_sim'])

        choice_errs = merged[merged['stat_name'] == 'rejection_rate']['abs_error']
        rt_errs = merged[merged['stat_name'].str.startswith('rt_')]['abs_error']

        choice_mae = float(choice_errs.mean()) if not choice_errs.empty else np.nan
        choice_max = float(choice_errs.max()) if not choice_errs.empty else np.nan
        rt_mae = float(rt_errs.mean()) if not rt_errs.empty else np.nan
        rt_max = float(rt_errs.max()) if not rt_errs.empty else np.nan

        is_adequate = (
            (not np.isnan(choice_mae)) and
            (choice_mae <= CFG.ppc_choice_mae_max) and
            (choice_max <= CFG.ppc_choice_max_err) and
            (not np.isnan(rt_mae)) and
            (rt_mae <= CFG.ppc_rt_quantile_mae_max) and
            (rt_max <= CFG.ppc_rt_quantile_max_err)
        )

        self.logger.info(
            f"  PPC Adequacy: Choice MAE={choice_mae:.4f} "
            f"(≤{CFG.ppc_choice_mae_max}), "
            f"RT MAE={rt_mae:.4f} (≤{CFG.ppc_rt_quantile_mae_max}) "
            f"-> {'PASS' if is_adequate else 'FAIL'}"
        )

        # Cleanup
        del model, ppc_data
        gc.collect()

        return {
            'choice_mae': choice_mae,
            'choice_max_err': choice_max,
            'rt_mae': rt_mae,
            'rt_max_err': rt_max,
            'pass': is_adequate
        }

    # -----------------------------------------------------------------
    # POSTERIOR PARAMETER MANIFEST GENERATION
    # -----------------------------------------------------------------
    def _generate_posterior_manifest(
        self, infdata: az.InferenceData, model_name: str
    ):
        """
        Categorizes posterior parameters into a structured manifest
        (family, level, focal/nuisance) for downstream audit and
        targeted diagnostics in Step 4.
        """
        all_vars = list(infdata.posterior.data_vars.keys())
        focal_params = CFG.identify_focal_parameters(all_vars)

        records = []
        for var in all_vars:
            family = var.split('_')[0] if '_' in var else var
            if family not in ['v', 'a', 't', 'z']:
                family = 'other'

            level = (
                'subject' if '_subj' in var
                else ('sd' if var.endswith('_std') or var.endswith('_var') else 'group')
            )
            is_focal = var in focal_params

            records.append({
                'variable_name': var,
                'family': family,
                'level': level,
                'focal': is_focal
            })

        df_manifest = pd.DataFrame(records)
        out_path = PATHS['audit'] / f"posterior_manifest_{model_name}.csv"
        df_manifest.to_csv(out_path, index=False)

    # -----------------------------------------------------------------
    # SINGLE MODEL PROCESSING
    # -----------------------------------------------------------------
    def process_model(self, model_name: str):
        """
        Processes a single model: loads InferenceData, generates
        diagnostics, computes PPC adequacy, exports manifests.
        """
        self.logger.info(f"\n{'─'*50}")
        self.logger.info(f"Processing: [{model_name.upper()}]")
        self.logger.info(f"{'─'*50}")

        # Check manifest for this model's status
        model_row = self.manifest[self.manifest['model_name'] == model_name]
        if model_row.empty:
            self.logger.warning(f"  [{model_name}] Not found in manifest. Skipping.")
            return

        # Load InferenceData from .nc file
        nc_path = PATHS['models'] / f"hddm_{model_name}.nc"
        if not nc_path.exists():
            self.logger.warning(
                f"  [{model_name}] NetCDF not found at {nc_path.name}. Skipping."
            )
            return

        infdata = az.from_netcdf(str(nc_path))
        self.logger.info(
            f"  Loaded InferenceData: {list(infdata.groups())}"
        )

        # 1. Generate posterior parameter manifest
        self._generate_posterior_manifest(infdata, model_name)

        # 2. Export ArviZ summary (reusable by Step 4)
        summary_df = az.summary(infdata, round_to=4, hdi_prob=0.95)
        focal_params = CFG.identify_focal_parameters(summary_df.index.tolist())
        summary_df['param_class'] = [
            'focal' if p in focal_params else 'nuisance'
            for p in summary_df.index
        ]
        summary_path = PATHS['audit'] / f"summary_{model_name}.csv"
        summary_df.to_csv(summary_path)
        self.logger.info(f"  ArviZ summary exported: {summary_path.name}")

        # 3. Generate diagnostic visualizations
        self._generate_trace_plots(infdata, model_name)
        self._generate_rank_plots(infdata, model_name)
        self._generate_ppc_density_plots(infdata, model_name)

        # 4. Compute quantitative PPC adequacy
        ppc_metrics = self._compute_ppc_adequacy_from_hddm(model_name)

        # 5. Update manifest with PPC results
        ppc_record_path = PATHS['ppc'] / f"ppc_metrics_{model_name}.json"
        with open(ppc_record_path, 'w', encoding='utf-8') as f:
            json.dump(
                {
                    'config_hash': self.active_hash,
                    'model_name': model_name,
                    **ppc_metrics
                },
                f, indent=2
            )

        # Cleanup
        del infdata
        gc.collect()

    # -----------------------------------------------------------------
    # MASTER PIPELINE
    # -----------------------------------------------------------------
    def run(self):
        """Orchestrates PPC audit across all fitted model architectures."""
        for model_name in CFG.final_all_models:
            self.process_model(model_name)

        # Export aggregated PPC statistics (long-format)
        if self.observed_stats:
            df_obs = pd.DataFrame(self.observed_stats)
            df_obs.to_csv(
                PATHS['ppc'] / "observed_summary_long.csv", index=False
            )
        if self.ppc_stats:
            df_sim = pd.DataFrame(self.ppc_stats)
            df_sim.to_csv(
                PATHS['ppc'] / "ppc_summary_long.csv", index=False
            )

        self.logger.info(f"\n{'='*70}")
        self.logger.info(
            f"STEP 3 COMPLETE: PPC audit finished for "
            f"{len(CFG.final_all_models)} models."
        )
        self.logger.info(
            f"Observed stats: {len(self.observed_stats)} records | "
            f"Simulated stats: {len(self.ppc_stats)} records"
        )
        self.logger.info(f"{'='*70}")


# =============================================================================
# PIPELINE EXECUTION
# =============================================================================
if __name__ == "__main__":
    engine = PPCAuditEngine()
    engine.run()

# Step 4: Convergence Diagnostics and Model Selection

In [ ]:
# -*- coding: utf-8 -*-
"""
=============================================================================
STEP 4: Four-Level Diagnostic Funnel & Optimal Model Selection
=============================================================================
Description:
  - Implements a rigorous Four-Level Evaluation Funnel:
    Level 1 (Technical):    Verifies artifact existence (.nc, .hddm).
    Level 2 (Convergence):  Evaluates stratified MCMC health from ArviZ
                            InferenceData (Vehtari et al., 2021).
    Level 3 (PPC Adequacy): Checks absolute goodness-of-fit via choice
                            and RT quantile MAE thresholds from Step 3.
    Level 4 (Relative):     Ranks surviving models by LOO-CV (primary) and
                            DIC (fallback).
  - Automatically selects the optimal model based on parsimony-weighted
    comparison among all models that pass Levels 1-3.
  - Generates publication-ready diagnostic visualizations for the winning
    model (trace plots, posterior pair plots).
  - Exports a structured audit trail stamped with the active configuration
    hash for downstream consumption by Steps 5-6.
  - References: Pan et al. (2025), Table 5; Spiegelhalter et al. (2002);
    Vehtari et al. (2017).
=============================================================================
"""

import gc
import json
import logging
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional

import numpy as np
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt
import seaborn as sns
import hddm

# -----------------------------------------------------------------------------
# CONFIGURATION IMPORT & ENVIRONMENT SETUP
# -----------------------------------------------------------------------------
try:
    from hddm_config import CFG, validate_pipeline_lineage
except ImportError:
    raise ImportError(
        "CRITICAL ERROR: 'hddm_config.py' not found. "
        "Execute Step 2a to generate the global configuration SSOT."
    )

PATHS = CFG.initialize_directories()

# Academic aesthetics
sns.set_theme(style="ticks")
plt.rcParams.update({
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'legend.frameon': False
})


# =============================================================================
# LOGGING SETUP
# =============================================================================
def _setup_funnel_logger() -> logging.Logger:
    """Initializes dual-sink logging for the diagnostic funnel."""
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    logger = logging.getLogger(f'hddm_funnel_{ts}')
    logger.handlers = []
    logger.setLevel(logging.INFO)

    fmt = logging.Formatter(
        '%(asctime)s - %(levelname)s - %(message)s', datefmt='%H:%M:%S'
    )
    fh = logging.FileHandler(
        PATHS['audit'] / f'model_selection_funnel_{ts}.log', encoding='utf-8'
    )
    ch = logging.StreamHandler()
    fh.setFormatter(fmt)
    ch.setFormatter(fmt)

    logger.addHandler(fh)
    logger.addHandler(ch)
    return logger


# =============================================================================
# CORE CLASS: FOUR-LEVEL DIAGNOSTIC FUNNEL
# =============================================================================
class DiagnosticFunnelEngine:
    """
    Executes a hierarchical model evaluation funnel that progressively
    filters model architectures through increasingly stringent criteria,
    then selects the optimal model among survivors based on LOO-CV or DIC.
    """

    def __init__(self):
        self.logger = _setup_funnel_logger()
        self.active_hash = validate_pipeline_lineage(PATHS, self.logger)
        self.manifest = self._load_manifest()

        self.comparison_records: List[Dict] = []

        self.logger.info("=" * 70)
        self.logger.info(
            f"FOUR-LEVEL DIAGNOSTIC FUNNEL INITIATED "
            f"(Mode: {CFG.run_mode.upper()})"
        )
        self.logger.info(f"Active Lineage Hash: {self.active_hash[:24]}...")
        self.logger.info(
            f"Stratified Criteria: "
            f"Focal R-hat≤{CFG.rhat_focal}, ESS≥{CFG.ess_bulk_focal} | "
            f"Nuisance R-hat≤{CFG.rhat_nuisance}, ESS≥{CFG.ess_bulk_nuisance}"
        )
        self.logger.info(
            f"PPC Adequacy: Choice MAE≤{CFG.ppc_choice_mae_max}, "
            f"RT MAE≤{CFG.ppc_rt_quantile_mae_max}"
        )
        self.logger.info("=" * 70)

    def _load_manifest(self) -> pd.DataFrame:
        """Loads Step 2b model manifest for metadata lookup."""
        manifest_path = PATHS['manifests'] / "model_manifest.csv"
        if manifest_path.exists():
            return pd.read_csv(manifest_path)
        self.logger.warning("Model manifest not found. Proceeding with discovery.")
        return pd.DataFrame()

    # -----------------------------------------------------------------
    # LEVEL 1: TECHNICAL ARTIFACT VERIFICATION
    # -----------------------------------------------------------------
    def evaluate_level1_technical(self, model_name: str) -> bool:
        """Verifies the existence of compiled ArviZ NetCDF and HDDM artifacts."""
        nc_file = PATHS['models'] / f"hddm_{model_name}.nc"
        hddm_file = PATHS['models'] / f"hddm_{model_name}.hddm"

        nc_ok = nc_file.exists()
        hddm_ok = hddm_file.exists()

        if not nc_ok:
            self.logger.warning(f"  L1 FAIL: Missing NetCDF -> {nc_file.name}")
        if not hddm_ok:
            self.logger.warning(f"  L1 FAIL: Missing HDDM -> {hddm_file.name}")

        return nc_ok and hddm_ok

    # -----------------------------------------------------------------
    # LEVEL 2: STRATIFIED CONVERGENCE DIAGNOSTICS
    # -----------------------------------------------------------------
    def evaluate_level2_convergence(
        self, infdata: az.InferenceData, model_name: str
    ) -> Dict[str, Any]:
        """
        Evaluates MCMC convergence directly from InferenceData using
        stratified criteria. Focal parameters (group-level effects) are
        held to stricter standards than nuisance parameters.
        """
        summary_df = az.summary(infdata, round_to=4, hdi_prob=0.95)

        # Defensive schema check
        required_cols = {'r_hat', 'ess_bulk'}
        missing = required_cols - set(summary_df.columns)
        if missing:
            return {
                'pass': False,
                'reason': f"ArviZ schema mismatch: missing {missing}",
                'max_rhat': np.nan,
                'min_ess_bulk': np.nan
            }

        # Classify parameters
        focal_params = CFG.identify_focal_parameters(summary_df.index.tolist())
        summary_df['param_class'] = [
            'focal' if p in focal_params else 'nuisance'
            for p in summary_df.index
        ]

        focal_df = summary_df[summary_df['param_class'] == 'focal']
        nuisance_df = summary_df[summary_df['param_class'] == 'nuisance']

        # Focal diagnostics
        f_rhat = focal_df['r_hat'].max() if not focal_df.empty else 1.0
        f_bulk = focal_df['ess_bulk'].min() if not focal_df.empty else float('inf')
        f_tail = (
            focal_df['ess_tail'].min()
            if ('ess_tail' in focal_df.columns and not focal_df.empty)
            else float('inf')
        )

        # Nuisance diagnostics
        n_rhat = nuisance_df['r_hat'].max() if not nuisance_df.empty else 1.0
        n_bulk = nuisance_df['ess_bulk'].min() if not nuisance_df.empty else float('inf')
        n_tail = (
            nuisance_df['ess_tail'].min()
            if ('ess_tail' in nuisance_df.columns and not nuisance_df.empty)
            else float('inf')
        )

        # Evaluate
        focal_ok = (
            f_rhat <= CFG.rhat_focal and
            f_bulk >= CFG.ess_bulk_focal and
            f_tail >= CFG.ess_tail_focal
        )
        nuisance_ok = (
            n_rhat <= CFG.rhat_nuisance and
            n_bulk >= CFG.ess_bulk_nuisance and
            n_tail >= CFG.ess_tail_nuisance
        )

        converged = focal_ok and nuisance_ok

        reason = "Passed" if converged else (
            f"Focal: R-hat={f_rhat:.3f},ESS_b={f_bulk:.0f},ESS_t={f_tail:.0f} | "
            f"Nuisance: R-hat={n_rhat:.3f},ESS_b={n_bulk:.0f}"
        )

        # Export focal parameter audit
        if not focal_df.empty:
            focal_df.to_csv(
                PATHS['audit'] / f"focal_parameter_audit_{model_name}.csv"
            )

        return {
            'pass': converged,
            'reason': reason,
            'f_rhat_max': f_rhat,
            'f_ess_bulk_min': f_bulk,
            'f_ess_tail_min': f_tail,
            'n_rhat_max': n_rhat,
            'n_ess_bulk_min': n_bulk,
            'n_ess_tail_min': n_tail
        }

    # -----------------------------------------------------------------
    # LEVEL 3: PPC ADEQUACY
    # -----------------------------------------------------------------
    def evaluate_level3_ppc(self, model_name: str) -> Dict[str, Any]:
        """
        Reads quantitative PPC metrics exported by Step 3 and evaluates
        against CFG adequacy thresholds.
        """
        ppc_metrics_path = PATHS['ppc'] / f"ppc_metrics_{model_name}.json"

        if not ppc_metrics_path.exists():
            self.logger.warning(
                f"  L3: PPC metrics not found for [{model_name}]. "
                f"Attempting long-format fallback..."
            )
            return self._evaluate_ppc_from_long_format(model_name)

        with open(ppc_metrics_path, 'r', encoding='utf-8') as f:
            ppc_data = json.load(f)

        is_adequate = ppc_data.get('pass', False)
        choice_mae = ppc_data.get('choice_mae', np.nan)
        rt_mae = ppc_data.get('rt_mae', np.nan)

        return {
            'pass': is_adequate,
            'choice_mae': choice_mae,
            'rt_mae': rt_mae
        }

    def _evaluate_ppc_from_long_format(
        self, model_name: str
    ) -> Dict[str, Any]:
        """
        Fallback PPC evaluation using the long-format CSV files
        (observed_summary_long.csv + ppc_summary_long.csv) if per-model
        JSON metrics are unavailable.
        """
        obs_path = PATHS['ppc'] / "observed_summary_long.csv"
        sim_path = PATHS['ppc'] / "ppc_summary_long.csv"

        if not obs_path.exists() or not sim_path.exists():
            return {'pass': False, 'choice_mae': np.nan, 'rt_mae': np.nan}

        df_obs = pd.read_csv(obs_path)
        df_sim = pd.read_csv(sim_path)

        df_sim_model = df_sim[df_sim['model_name'] == model_name]
        if df_sim_model.empty:
            return {'pass': False, 'choice_mae': np.nan, 'rt_mae': np.nan}

        # Aggregate simulated statistics
        merge_keys = ['emotion', 'stat_name']
        if 'response_type' in df_sim_model.columns:
            merge_keys.append('response_type')

        sim_agg = (
            df_sim_model
            .groupby(merge_keys)['stat_value']
            .mean()
            .reset_index()
            .rename(columns={'stat_value': 'sim_value'})
        )

        merged = pd.merge(
            df_obs[df_obs['source'] == 'observed'] if 'source' in df_obs.columns else df_obs,
            sim_agg,
            on=merge_keys,
            how='inner'
        )

        if 'stat_value' in merged.columns and 'sim_value' in merged.columns:
            merged['abs_error'] = np.abs(merged['stat_value'] - merged['sim_value'])
        else:
            return {'pass': False, 'choice_mae': np.nan, 'rt_mae': np.nan}

        choice_errs = merged[merged['stat_name'] == 'rejection_rate']['abs_error']
        rt_errs = merged[merged['stat_name'].str.startswith('rt_')]['abs_error']

        choice_mae = float(choice_errs.mean()) if not choice_errs.empty else np.nan
        rt_mae = float(rt_errs.mean()) if not rt_errs.empty else np.nan

        is_adequate = (
            (not np.isnan(choice_mae)) and
            (choice_mae <= CFG.ppc_choice_mae_max) and
            (not np.isnan(rt_mae)) and
            (rt_mae <= CFG.ppc_rt_quantile_mae_max)
        )

        return {'pass': is_adequate, 'choice_mae': choice_mae, 'rt_mae': rt_mae}

    # -----------------------------------------------------------------
    # LEVEL 4: RELATIVE MODEL COMPARISON (DIC + optional LOO/WAIC)
    # -----------------------------------------------------------------
    def compute_model_comparison_metrics(
        self, model_name: str, infdata: az.InferenceData
    ) -> Dict[str, float]:
        """
        Extracts DIC from the HDDM model object (always available).
        Optionally computes LOO-CV and WAIC from InferenceData when
        the log_likelihood group is present.
        """
        metrics = {'dic': float('inf'), 'loo': np.nan, 'waic': np.nan}

        # DIC: always available via HDDM model object
        hddm_path = PATHS['models'] / f"hddm_{model_name}.hddm"
        if hddm_path.exists():
            try:
                model = hddm.load(str(hddm_path))
                metrics['dic'] = float(model.dic)
                del model
                gc.collect()
            except Exception as e:
                self.logger.warning(f"  DIC extraction failed: {e}")

        # LOO/WAIC: only when log_likelihood is available in InferenceData
        if hasattr(infdata, 'log_likelihood'):
            try:
                loo_result = az.loo(infdata)
                metrics['loo'] = float(loo_result.elpd_loo)  # Note: elpd (higher = better)
                self.logger.info(
                    f"  LOO-CV: elpd={metrics['loo']:.2f}"
                )
            except Exception as e:
                self.logger.warning(f"  LOO-CV computation failed: {e}")

            try:
                waic_result = az.waic(infdata)
                metrics['waic'] = float(waic_result.elpd_waic)
                self.logger.info(
                    f"  WAIC:   elpd={metrics['waic']:.2f}"
                )
            except Exception as e:
                self.logger.warning(f"  WAIC computation failed: {e}")

        return metrics

    # -----------------------------------------------------------------
    # WINNING MODEL DIAGNOSTICS
    # -----------------------------------------------------------------
    def generate_winner_diagnostics(
        self, model_name: str, infdata: az.InferenceData
    ):
        """
        Generates targeted publication-ready diagnostics for the winning
        model: trace plots, rank plots, and posterior pair plots.
        """
        self.logger.info(
            f"\nGenerating supplementary diagnostics for winner: "
            f"[{model_name.upper()}]"
        )

        all_vars = list(infdata.posterior.data_vars.keys())
        focal_vars = CFG.identify_focal_parameters(all_vars)

        if not focal_vars:
            self.logger.warning("No focal parameters found for diagnostics.")
            return

        # 1. Trace plot (focal parameters only)
        try:
            az.plot_trace(
                infdata,
                var_names=focal_vars,
                compact=True,
                figsize=(12, 2.5 * len(focal_vars))
            )
            plt.tight_layout()
            plt.savefig(
                PATHS['figures_supp'] / f"winner_trace_{model_name}.pdf",
                dpi=300, bbox_inches='tight'
            )
            plt.close()
            self.logger.info("  Winner trace plot saved.")
        except Exception as e:
            self.logger.warning(f"  Trace plot failed: {e}")

        # 2. Rank plot
        try:
            az.plot_rank(
                infdata,
                var_names=focal_vars,
                kind='vlines',
                vlines_kwargs={'lw': 0},
                marker_vlines_kwargs={'lw': 2}
            )
            plt.tight_layout()
            plt.savefig(
                PATHS['figures_supp'] / f"winner_rank_{model_name}.pdf",
                dpi=300, bbox_inches='tight'
            )
            plt.close()
            self.logger.info("  Winner rank plot saved.")
        except Exception as e:
            self.logger.warning(f"  Rank plot failed: {e}")

        # 3. Posterior pair plot (correlation structure between focal params)
        try:
            # Limit to reasonable number of parameters for readability
            pair_vars = focal_vars[:8] if len(focal_vars) > 8 else focal_vars
            az.plot_pair(
                infdata,
                var_names=pair_vars,
                kind='kde',
                marginals=True,
                figsize=(12, 12)
            )
            plt.tight_layout()
            plt.savefig(
                PATHS['figures_supp'] / f"winner_pair_{model_name}.pdf",
                dpi=300, bbox_inches='tight'
            )
            plt.close()
            self.logger.info("  Winner pair plot saved.")
        except Exception as e:
            self.logger.warning(f"  Pair plot failed: {e}")

    # -----------------------------------------------------------------
    # FULL FUNNEL EXECUTION
    # -----------------------------------------------------------------
    def execute_funnel(self):
        """
        Iterates over all model architectures, passing each through
        the four-level evaluation funnel.
        """
        for model_name in CFG.final_all_models:
            self.logger.info(f"\n{'─'*50}")
            self.logger.info(f"Evaluating: [{model_name.upper()}]")
            self.logger.info(f"{'─'*50}")

            tier = CFG.mcmc_protocols.get(model_name, {}).get('tier', 'unknown')

            record = {
                'config_hash': self.active_hash,
                'model_name': model_name,
                'tier': tier,
                'L1_technical': False,
                'L2_converged': False,
                'L3_ppc_adequate': False,
                'dic': float('inf'),
                'loo_elpd': np.nan,
                'waic_elpd': np.nan,
                'f_rhat_max': np.nan,
                'f_ess_bulk_min': np.nan,
                'choice_mae': np.nan,
                'rt_mae': np.nan,
                'overall_pass': False,
                'rejection_reason': 'Not evaluated'
            }

            # LEVEL 1: Technical artifact check
            if not self.evaluate_level1_technical(model_name):
                record['rejection_reason'] = 'L1: Missing artifacts'
                self.comparison_records.append(record)
                self.logger.info(f"  REJECTED at Level 1 (Missing artifacts)")
                continue

            record['L1_technical'] = True

            # Load InferenceData for Levels 2-4
            nc_path = PATHS['models'] / f"hddm_{model_name}.nc"
            infdata = az.from_netcdf(str(nc_path))

            # LEVEL 2: Stratified convergence
            l2_result = self.evaluate_level2_convergence(infdata, model_name)
            record['L2_converged'] = l2_result['pass']
            record['f_rhat_max'] = l2_result.get('f_rhat_max', np.nan)
            record['f_ess_bulk_min'] = l2_result.get('f_ess_bulk_min', np.nan)

            if not l2_result['pass']:
                record['rejection_reason'] = f"L2: {l2_result['reason']}"
                self.comparison_records.append(record)
                self.logger.info(
                    f"  REJECTED at Level 2 (Convergence): {l2_result['reason']}"
                )
                del infdata
                gc.collect()
                continue

            self.logger.info(f"  Level 2 PASSED (Convergence OK)")

            # LEVEL 3: PPC Adequacy
            l3_result = self.evaluate_level3_ppc(model_name)
            record['L3_ppc_adequate'] = l3_result['pass']
            record['choice_mae'] = l3_result.get('choice_mae', np.nan)
            record['rt_mae'] = l3_result.get('rt_mae', np.nan)

            if not l3_result['pass']:
                record['rejection_reason'] = (
                    f"L3: PPC inadequate "
                    f"(choice_mae={l3_result['choice_mae']:.4f}, "
                    f"rt_mae={l3_result['rt_mae']:.4f})"
                )
                self.comparison_records.append(record)
                self.logger.info(
                    f"  REJECTED at Level 3 (PPC): {record['rejection_reason']}"
                )
                del infdata
                gc.collect()
                continue

            self.logger.info(
                f"  Level 3 PASSED (PPC: choice_mae={l3_result['choice_mae']:.4f})"
            )

            # LEVEL 4: Relative comparison metrics
            l4_metrics = self.compute_model_comparison_metrics(
                model_name, infdata
            )
            record['dic'] = l4_metrics['dic']
            record['loo_elpd'] = l4_metrics.get('loo', np.nan)
            record['waic_elpd'] = l4_metrics.get('waic', np.nan)
            record['overall_pass'] = True
            record['rejection_reason'] = 'Passed all levels'

            self.comparison_records.append(record)

            self.logger.info(
                f"  Level 4: DIC={l4_metrics['dic']:.2f}, "
                f"LOO={l4_metrics.get('loo', 'N/A')}, "
                f"WAIC={l4_metrics.get('waic', 'N/A')}"
            )
            self.logger.info(f"  STATUS: PASSED ALL LEVELS")

            del infdata
            gc.collect()

    # -----------------------------------------------------------------
    # OPTIMAL MODEL SELECTION & EXPORT
    # -----------------------------------------------------------------
    def select_optimal_model(self) -> Optional[str]:
        """
        Among models that passed all four levels, selects the optimal
        model based on highest LOO-CV elpd (primary criterion). If LOO-CV is
        unavailable, falls back to lowest DIC. Exports the full audit trail.
        """
        df = pd.DataFrame(self.comparison_records)

        # Identify eligible models (passed all levels)
        eligible = df[df['overall_pass'] == True].copy()

        if eligible.empty:
            self.logger.error(
                "\nCRITICAL: No models passed the four-level funnel."
            )
            # Export full comparison for debugging
            df.to_csv(
                PATHS['tables_main'] / "model_comparison_summary.csv",
                index=False
            )
            raise RuntimeError(
                "Funnel collapsed. All models failed. "
                "Check audit logs for details."
            )

        # Primary criterion: highest LOO-CV elpd, fallback to lowest DIC
        if 'loo_elpd' in eligible.columns and not eligible['loo_elpd'].isna().all():
            winner_idx = eligible['loo_elpd'].idxmax()
            winning_model = df.at[winner_idx, 'model_name']
            win_reason = f"WINNER (Highest LOO-CV among {len(eligible)} survivors)"
            self.logger.info("  Selection Criterion: LOO-CV (Primary)")
        else:
            winner_idx = eligible['dic'].idxmin()
            winning_model = df.at[winner_idx, 'model_name']
            win_reason = f"WINNER (Lowest DIC among {len(eligible)} survivors)"
            self.logger.info("  Selection Criterion: DIC (Fallback)")

        # Mark winner in the full table
        df['Is_Winner'] = False
        df.at[winner_idx, 'Is_Winner'] = True
        df.at[winner_idx, 'rejection_reason'] = win_reason

        # Sort: passed models first (by DIC to keep output predictable), then failed models
        df = df.sort_values(
            by=['overall_pass', 'dic'],
            ascending=[False, True]
        )

        # Export full comparison summary
        df.to_csv(
            PATHS['tables_main'] / "model_comparison_summary.csv",
            index=False
        )

        # Export focused audit trail (for Step 5/6 consumption)
        audit_record = df[df['Is_Winner'] == True].copy()
        audit_record.to_csv(
            PATHS['audit'] / "final_model_selection_audit.csv",
            index=False
        )

        # Console report
        self.logger.info(f"\n{'='*70}")
        if 'loo_elpd' in eligible.columns and not pd.isna(df.at[winner_idx, 'loo_elpd']):
            self.logger.info(
                f"OPTIMAL MODEL: [{winning_model.upper()}] "
                f"(LOO-CV elpd={df.at[winner_idx, 'loo_elpd']:.2f})"
            )
        else:
            self.logger.info(
                f"OPTIMAL MODEL: [{winning_model.upper()}] "
                f"(DIC={df.at[winner_idx, 'dic']:.2f})"
            )
        self.logger.info(f"{'='*70}")

        # Consistency check reporting
        if not eligible['loo_elpd'].isna().all() and not eligible['dic'].isna().all():
            dic_best = eligible.loc[eligible['dic'].idxmin(), 'model_name']
            consistent = (dic_best == winning_model)
            self.logger.info(
                f"  DIC best: [{dic_best}] "
                f"({'CONSISTENT' if consistent else 'INCONSISTENT'} with LOO-CV)"
            )
        
        # Print summary table
        display_cols = [
            'model_name', 'tier', 'L1_technical', 'L2_converged',
            'L3_ppc_adequate', 'overall_pass', 'loo_elpd', 'dic',
            'f_rhat_max', 'rejection_reason'
        ]
        available = [c for c in display_cols if c in df.columns]
        print("\n--- Model Comparison Funnel Results ---")
        print(df[available].to_string(index=False))

        # Generate diagnostics for the winner
        nc_path = PATHS['models'] / f"hddm_{winning_model}.nc"
        if nc_path.exists():
            infdata = az.from_netcdf(str(nc_path))
            self.generate_winner_diagnostics(winning_model, infdata)

            # Optional: ArviZ model comparison table (if LOO available)
            if hasattr(infdata, 'log_likelihood'):
                self._try_arviz_compare(eligible)

            del infdata
            gc.collect()

        return winning_model

    def _try_arviz_compare(self, eligible_df: pd.DataFrame):
        """
        Attempts to produce ArviZ's model comparison table using
        az.compare() for all eligible models that have log_likelihood.
        This is a best-effort operation; failures are logged but do not
        block the pipeline.
        """
        compare_dict = {}
        for _, row in eligible_df.iterrows():
            model_name = row['model_name']
            nc_path = PATHS['models'] / f"hddm_{model_name}.nc"
            if nc_path.exists():
                idata = az.from_netcdf(str(nc_path))
                if hasattr(idata, 'log_likelihood'):
                    compare_dict[model_name] = idata

        if len(compare_dict) < 2:
            return

        try:
            # LOO comparison
            loo_compare = az.compare(compare_dict, ic='loo')
            loo_compare.to_csv(
                PATHS['tables_main'] / "arviz_loo_comparison.csv"
            )
            self.logger.info("  ArviZ LOO comparison table exported.")

            # WAIC comparison
            waic_compare = az.compare(compare_dict, ic='waic')
            waic_compare.to_csv(
                PATHS['tables_main'] / "arviz_waic_comparison.csv"
            )
            self.logger.info("  ArviZ WAIC comparison table exported.")

        except Exception as e:
            self.logger.warning(f"  ArviZ compare failed: {e}")
        finally:
            # Cleanup loaded InferenceData objects
            for idata in compare_dict.values():
                del idata
            gc.collect()


# =============================================================================
# PIPELINE EXECUTION
# =============================================================================
if __name__ == "__main__":
    funnel = DiagnosticFunnelEngine()
    funnel.execute_funnel()
    winning_model = funnel.select_optimal_model()

# Step 5: Statistical Inference and Publication-Ready Visualization

In [ ]:
# -*- coding: utf-8 -*-
"""
=============================================================================
STEP 5: Bayesian Posterior Inference, HDI/ROPE Evaluation & Publication
         Visualization (Physical Scale Adaptation)
=============================================================================
Description:
  - Loads the optimal model's InferenceData from Step 4's selection.
  - Enforces cryptographic lineage validation against the active
    configuration hash.
  - CRITICAL METHODOLOGICAL STEP: Reconstructs absolute posterior distributions
    and physical contrasts by applying proper Inverse Link Functions
    (exp for 'a' and 't', expit for 'z') BEFORE computing inferential metrics.
    This guarantees that HDI and ROPE are evaluated on the ecologically valid
    physical scale (e.g., reaction time in seconds, probability) rather than
    the latent linear predictor scale.
  - Computes rigorous Bayesian inferential metrics per physical contrast:
    * 95% Highest Density Interval (HDI)
    * Probability of Direction (Pd)
    * Region of Practical Equivalence (ROPE) overlap (Kruschke, 2018)
    * HDI excludes zero (strict significance indicator)
  - Generates publication-ready visualizations:
    * ArviZ az.plot_posterior() with ROPE annotation on physical contrasts.
    * Raincloud plots (half-violin + boxplot + jitter) of absolute posteriors.
    * Overlapping KDE contrast plots with HDI bars.
  - Exports strict academic reporting tables stamped with lineage hash.
=============================================================================
"""

import os
import gc
import json
import logging
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import expit

# Conditional import: ptitprince provides Raincloud plots but may not be
# available in all environments. Fall back to seaborn violin if missing.
try:
    import ptitprince as pt
    HAS_PTITPRINCE = True
except ImportError:
    HAS_PTITPRINCE = False

# -----------------------------------------------------------------------------
# CONFIGURATION IMPORT & ENVIRONMENT SETUP
# -----------------------------------------------------------------------------
try:
    from hddm_config import (
        CFG, validate_pipeline_lineage, identify_winning_model
    )
except ImportError:
    raise ImportError(
        "CRITICAL ERROR: 'hddm_config.py' not found. "
        "Execute Step 2a to generate the global configuration SSOT."
    )

PATHS = CFG.initialize_directories()

# Apply strict academic visualization aesthetics (Nature / APA style)
sns.set_theme(style="ticks")
plt.rcParams.update({
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'axes.labelsize': 14,
    'legend.fontsize': 11,
    'legend.frameon': False
})


# =============================================================================
# LOGGING SETUP
# =============================================================================
def _setup_inference_logger() -> logging.Logger:
    """Initializes dual-sink logging for the statistical inference module."""
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    logger = logging.getLogger(f'hddm_inference_{ts}')
    logger.handlers = []
    logger.setLevel(logging.INFO)

    fmt = logging.Formatter(
        '%(asctime)s - %(levelname)s - %(message)s', datefmt='%H:%M:%S'
    )
    fh = logging.FileHandler(
        PATHS['audit'] / f'statistical_inference_{ts}.log', encoding='utf-8'
    )
    ch = logging.StreamHandler()
    fh.setFormatter(fmt)
    ch.setFormatter(fmt)

    logger.addHandler(fh)
    logger.addHandler(ch)
    return logger


# =============================================================================
# CORE CLASS: BAYESIAN INFERENCE & VISUALIZATION ENGINE
# =============================================================================
class BayesianInferenceVisualizer:
    """
    Orchestrates posterior extraction, inverse-link transformations, HDI/ROPE
    hypothesis testing on the physical scale, and publication-quality
    visualization for the optimal HDDM architecture.
    """

    def __init__(self):
        self.logger = _setup_inference_logger()
        self.active_hash = validate_pipeline_lineage(PATHS, self.logger)
        self.winning_model = identify_winning_model(PATHS)
        self.infdata: Optional[az.InferenceData] = None
        self.stats_records: List[Dict] = []
        
        # Cache physical contrasts for ArviZ native plotting
        self.physical_contrasts_dict: Dict[str, np.ndarray] = {}

        # Cache design variables from SSOT
        self.baseline = CFG.baseline_condition
        self.emotions = CFG.emotion_order
        self.colors = CFG.colors
        self.labels = CFG.display_labels

        self.logger.info("=" * 70)
        self.logger.info(
            f"BAYESIAN INFERENCE ENGINE INITIATED "
            f"(Mode: {CFG.run_mode.upper()})"
        )
        self.logger.info(f"Lineage Hash: {self.active_hash[:24]}...")
        self.logger.info(f"Optimal Model: [{self.winning_model.upper()}]")
        self.logger.info("=" * 70)

        self._load_inference_data()

    def _load_inference_data(self):
        """Loads the winning model's InferenceData from NetCDF."""
        nc_path = PATHS['models'] / f"hddm_{self.winning_model}.nc"
        if not nc_path.exists():
            raise FileNotFoundError(
                f"InferenceData not found: {nc_path.name}. "
                f"Ensure Steps 2b-4 completed successfully."
            )

        self.infdata = az.from_netcdf(str(nc_path))
        self.logger.info(
            f"Loaded InferenceData: groups={list(self.infdata.groups())}"
        )

    # -----------------------------------------------------------------
    # ROPE BOUNDARY DEFINITIONS (PHYSICAL SCALE)
    # -----------------------------------------------------------------
    @staticmethod
    def _define_rope_for_param(param_family: str) -> Tuple[float, float]:
        """
        Defines the Region of Practical Equivalence boundaries per DDM
        parameter family. These bounds are evaluated on the PHYSICAL scale
        (absolute shift in drift rate, threshold, seconds, or probability).
        """
        rope_mapping = {
            'v': (-0.20, 0.20),    # Shift in drift rate (identity)
            'a': (-0.05, 0.05),    # Absolute shift in threshold magnitude
            't': (-0.01, 0.01),    # Shift in non-decision time (seconds)
            'z': (-0.02, 0.02)     # Shift in starting point (probability ratio)
        }
        return rope_mapping.get(param_family, (-0.10, 0.10))

    # -----------------------------------------------------------------
    # INVERSE LINK FUNCTION
    # -----------------------------------------------------------------
    @staticmethod
    def _apply_inverse_link(param_family: str, trace: np.ndarray) -> np.ndarray:
        """
        Applies the HDDM internal inverse link function to the linear predictor
        trace. Employs clipping to prevent computational overflow in extreme
        MCMC outlier draws.
        """
        if param_family == 'v':
            return trace
        elif param_family == 'a':
            # Threshold is log-linked in HDDM. Clip to avoid exp() overflow.
            safe_trace = np.clip(trace, -10.0, 4.0)
            return np.exp(safe_trace)
        elif param_family == 't':
            # NDT is log-linked.
            safe_trace = np.clip(trace, -10.0, 2.0)
            return np.exp(safe_trace)
        elif param_family == 'z':
            # Starting point bias is logistic-linked.
            return expit(trace)
        return trace

    # -----------------------------------------------------------------
    # POSTERIOR TRACE EXTRACTION (from InferenceData)
    # -----------------------------------------------------------------
    def _get_flattened_traces(self) -> pd.DataFrame:
        """
        Extracts posterior samples from InferenceData, flattening the
        (chain, draw) dimensions into a single sample axis.
        Returns a DataFrame with columns = parameter names.
        """
        post = self.infdata.posterior
        traces = {}
        for var_name in post.data_vars:
            traces[var_name] = post[var_name].values.flatten()
        return pd.DataFrame(traces)

    # -----------------------------------------------------------------
    # RECONSTRUCTION & INFERENTIAL COMPUTATION
    # -----------------------------------------------------------------
    def reconstruct_and_evaluate(
        self, param_family: str
    ) -> Tuple[pd.DataFrame, List[Dict]]:
        """
        Reconstructs absolute posteriors and evaluates contrast HDI and
        ROPE for each non-baseline condition within a given DDM parameter
        family, strictly operating on the physical scale.

        Methodology:
          1. Base_LP = Intercept
          2. Base_Physical = InverseLink(Base_LP)
          3. Cond_LP = Intercept + TreatmentContrast
          4. Cond_Physical = InverseLink(Cond_LP)
          5. Physical_Contrast = Cond_Physical - Base_Physical
          6. Evaluate HDI/ROPE on Physical_Contrast.

        Returns:
          reconstructed_df: Long-format DataFrame [Emotion, Posterior_Value].
          inference_records: Per-contrast statistics (HDI, Pd, ROPE).
        """
        traces = self._get_flattened_traces()
        reconstructed_data = []
        inference_records = []

        intercept_col = f"{param_family}_Intercept"
        if intercept_col not in traces.columns:
            self.logger.warning(
                f"Intercept '{intercept_col}' not found. "
                f"Skipping family '{param_family}'."
            )
            return pd.DataFrame(), []

        base_lp_trace = traces[intercept_col].values
        base_phys_trace = self._apply_inverse_link(param_family, base_lp_trace)

        # Baseline condition absolute posterior
        reconstructed_data.append(pd.DataFrame({
            "Emotion": self.baseline,
            "Posterior_Value": base_phys_trace
        }))

        rope_bounds = self._define_rope_for_param(param_family)

        for em in self.emotions:
            if em == self.baseline:
                continue

            # Treatment contrast column lookup
            contrast_col = (
                f"{param_family}_C(emotion, "
                f"Treatment('{self.baseline}'))[T.{em}]"
            )
            if contrast_col not in traces.columns:
                contrast_col = f"{param_family}_C(emotion)[T.{em}]"

            if contrast_col not in traces.columns:
                self.logger.warning(
                    f"Contrast column not found for "
                    f"{param_family} x {em}."
                )
                continue

            contrast_lp_trace = traces[contrast_col].values
            cond_lp_trace = base_lp_trace + contrast_lp_trace
            cond_phys_trace = self._apply_inverse_link(param_family, cond_lp_trace)
            
            # The critical physical contrast
            phys_contrast_trace = cond_phys_trace - base_phys_trace
            
            # Cache for ArviZ native plotting
            safe_var_name = f"{param_family}_{em}_vs_{self.baseline}"
            self.physical_contrasts_dict[safe_var_name] = phys_contrast_trace

            # --- Bayesian Inferential Metrics on the PHYSICAL CONTRAST ---
            hdi_bounds = az.hdi(phys_contrast_trace, hdi_prob=0.95)
            median_diff = float(np.median(phys_contrast_trace))
            mean_diff = float(np.mean(phys_contrast_trace))

            # Probability of Direction (Pd)
            p_greater = float(np.mean(phys_contrast_trace > 0))
            pd_val = max(p_greater, 1.0 - p_greater)

            # ROPE overlap
            rope_overlap = float(np.mean(
                (phys_contrast_trace >= rope_bounds[0]) &
                (phys_contrast_trace <= rope_bounds[1])
            ))

            # HDI excludes zero
            hdi_excludes_zero = (hdi_bounds[0] > 0) or (hdi_bounds[1] < 0)

            inference_records.append({
                'config_hash': self.active_hash,
                'model_name': self.winning_model,
                'Parameter_Family': param_family,
                'Condition': em,
                'Contrast_vs': self.baseline,
                'Physical_Contrast_Mean': mean_diff,
                'Physical_Contrast_Median': median_diff,
                'HDI_95_Low': float(hdi_bounds[0]),
                'HDI_95_High': float(hdi_bounds[1]),
                'HDI_Excludes_Zero': hdi_excludes_zero,
                'Pd': pd_val,
                'P_Greater_Than_Zero': p_greater,
                'ROPE_Low': rope_bounds[0],
                'ROPE_High': rope_bounds[1],
                'ROPE_Overlap_Pct': rope_overlap * 100
            })

            reconstructed_data.append(pd.DataFrame({
                "Emotion": em,
                "Posterior_Value": cond_phys_trace
            }))

        if reconstructed_data:
            return pd.concat(reconstructed_data, ignore_index=True), inference_records
        return pd.DataFrame(), inference_records

    def compute_all_posterior_statistics(self):
        """
        Iterates over all DDM parameter families present in the winning
        model's varying specification and computes the full inferential table.
        """
        self.logger.info(
            "\nComputing Bayesian HDI/ROPE hypothesis testing metrics "
            "(Physical Scale)..."
        )

        model_lower = self.winning_model.lower()
        families = [p for p in ['v', 'a', 't', 'z'] if p in model_lower]

        all_records = []
        for family in families:
            _, records = self.reconstruct_and_evaluate(family)
            if records:
                all_records.extend(records)
                self.logger.info(
                    f"  [{family.upper()}] {len(records)} physical contrasts evaluated."
                )

        self.stats_records = all_records
        df_stats = pd.DataFrame(all_records)

        if not df_stats.empty:
            out_path = (
                PATHS['tables_main'] /
                f"bayesian_inference_physical_summary_{self.winning_model}.csv"
            )
            df_stats.to_csv(out_path, index=False)
            self.logger.info(
                f"Inference table: {out_path.name} ({len(df_stats)} records)"
            )

            # Print summary to console
            print("\n--- Bayesian Inference Summary (Physical Scale) ---")
            display_cols = [
                'Parameter_Family', 'Condition', 'Physical_Contrast_Median',
                'HDI_95_Low', 'HDI_95_High', 'HDI_Excludes_Zero',
                'Pd', 'ROPE_Overlap_Pct'
            ]
            print(df_stats[display_cols].to_string(index=False))
        else:
            self.logger.warning("No valid contrasts found for any family.")

    # -----------------------------------------------------------------
    # VISUALIZATION 1: ArviZ Native Posterior + ROPE Plot
    # -----------------------------------------------------------------
    def render_arviz_posterior_rope(self):
        """
        Uses az.plot_posterior() with ROPE annotation on the PHYSICAL contrasts.
        We dynamically construct a pseudo-InferenceData object from the derived
        physical differences to leverage ArviZ's native aesthetics while
        displaying the correct physiological magnitudes.
        """
        self.logger.info("\nRendering ArviZ posterior + ROPE plots (Physical Scale)...")

        if not self.physical_contrasts_dict:
            self.logger.warning("No physical contrasts available for plotting.")
            return
            
        # Convert flat traces back to mock (chain, draw) structure for ArviZ
        # We assume 1 chain for the mock object just to satisfy ArviZ requirements
        mock_posteriors = {}
        for var_name, trace in self.physical_contrasts_dict.items():
            mock_posteriors[var_name] = np.expand_dims(trace, axis=0)
            
        mock_infdata = az.from_dict(posterior=mock_posteriors)

        for var_name in mock_posteriors.keys():
            family = var_name.split('_')[0]
            rope = self._define_rope_for_param(family)

            try:
                ax = az.plot_posterior(
                    mock_infdata,
                    var_names=[var_name],
                    hdi_prob=0.95,
                    rope=rope,
                    figsize=(8, 4),
                    textsize=12
                )
                plt.title(f"Physical Contrast: {var_name}", fontsize=14, pad=10)
                plt.tight_layout()

                out_path = (
                    PATHS['figures_main'] /
                    f"posterior_rope_physical_{var_name}.pdf"
                )
                plt.savefig(out_path, dpi=300, bbox_inches='tight')
                plt.close()
                self.logger.info(f"  Saved: {out_path.name}")
            except Exception as e:
                self.logger.warning(
                    f"  az.plot_posterior failed for {var_name}: {e}"
                )

    # -----------------------------------------------------------------
    # VISUALIZATION 2: Raincloud Plots (Absolute Posteriors)
    # -----------------------------------------------------------------
    def render_raincloud(self, param_family: str):
        """
        Generates a Raincloud plot showing reconstructed absolute posterior 
        distributions (Physical Scale) per condition.
        """
        df_recon, _ = self.reconstruct_and_evaluate(param_family)
        if df_recon.empty:
            return

        df_recon["Emotion_Label"] = df_recon["Emotion"].map(self.labels)
        ordered_labels = [
            self.labels[em] for em in self.emotions
            if em in df_recon["Emotion"].unique()
        ]
        ordered_colors = [
            self.colors[em] for em in self.emotions
            if em in df_recon["Emotion"].unique()
        ]

        fig, ax = plt.subplots(figsize=(11, 7))

        if HAS_PTITPRINCE:
            pt.RainCloud(
                x="Emotion_Label", y="Posterior_Value", data=df_recon,
                palette=ordered_colors, bw=0.2, width_viol=0.6,
                ax=ax, orient="v", alpha=0.65, dodge=True,
                order=ordered_labels
            )
        else:
            sns.violinplot(
                x="Emotion_Label", y="Posterior_Value", data=df_recon,
                palette=ordered_colors, order=ordered_labels,
                inner="box", cut=0, ax=ax, alpha=0.6
            )
            sns.stripplot(
                x="Emotion_Label", y="Posterior_Value", data=df_recon,
                palette=ordered_colors, order=ordered_labels,
                size=1.5, alpha=0.3, jitter=True, ax=ax
            )

        # Baseline reference line
        baseline_median = df_recon[
            df_recon["Emotion"] == self.baseline
        ]["Posterior_Value"].median()
        ax.axhline(
            baseline_median, color='gray', linestyle='--',
            alpha=0.5, zorder=0,
            label=f'{self.labels[self.baseline]} Median'
        )

        # Annotate HDI and ROPE
        family_records = [
            r for r in self.stats_records
            if r['Parameter_Family'] == param_family
        ]
        for record in family_records:
            cond_label = self.labels.get(record['Condition'], '')
            if cond_label not in ordered_labels:
                continue
            idx = ordered_labels.index(cond_label)

            hdi_low = record['HDI_95_Low']
            hdi_high = record['HDI_95_High']
            rope_pct = record['ROPE_Overlap_Pct']

            cond_data = df_recon[
                df_recon["Emotion"] == record['Condition']
            ]["Posterior_Value"]
            y_top = cond_data.quantile(0.975) + 0.05 * abs(cond_data.std())

            annotation = (
                f"HDI: [{hdi_low:.3f}, {hdi_high:.3f}]\n"
                f"ROPE: {rope_pct:.1f}%"
            )
            ax.annotate(
                annotation, xy=(idx, y_top),
                fontsize=8, ha='center', va='bottom',
                bbox=dict(
                    boxstyle='round,pad=0.3', facecolor='white',
                    edgecolor='gray', alpha=0.8
                )
            )

        ax.set_title(
            f"Absolute Posterior Distributions: {param_family.upper()} "
            f"({self.winning_model.upper()})",
            fontsize=16, fontweight='bold', pad=20
        )
        ax.set_ylabel(
            f"Physical Parameter Value ({param_family})", fontsize=14
        )
        ax.set_xlabel("Experimental Condition", fontsize=14)
        sns.despine(trim=True)

        fig.tight_layout()
        fig.savefig(
            PATHS['figures_main'] /
            f"posterior_raincloud_physical_{param_family}_{self.winning_model}.pdf",
            dpi=300, bbox_inches='tight'
        )
        plt.close(fig)
        self.logger.info(
            f"  Raincloud saved: "
            f"posterior_raincloud_physical_{param_family}_{self.winning_model}.pdf"
        )

    # -----------------------------------------------------------------
    # VISUALIZATION 3: Contrast KDE with HDI Bars
    # -----------------------------------------------------------------
    def render_contrast_kde(self):
        """
        Generates overlapping KDE plots for Physical contrasts (difference
        from baseline) with HDI shading bars.
        """
        self.logger.info("\nRendering contrast KDE overview (Physical Scale)...")

        model_lower = self.winning_model.lower()
        families = [p for p in ['v', 'a', 't', 'z'] if p in model_lower]

        for family in families:
            family_records = [
                r for r in self.stats_records
                if r['Parameter_Family'] == family
            ]
            if not family_records:
                continue

            fig, ax = plt.subplots(figsize=(8, 5))
            ax.axvline(
                x=0, color='black', linestyle='--', linewidth=1.5,
                zorder=1, label="Null Effect"
            )

            y_offset = 0.0
            for record in family_records:
                cond = record['Condition']
                if cond not in self.colors:
                    continue

                var_name = f"{family}_{cond}_vs_{self.baseline}"
                if var_name not in self.physical_contrasts_dict:
                    continue

                contrast_vals = self.physical_contrasts_dict[var_name]
                color = self.colors[cond]
                label = self.labels.get(cond, cond)

                sns.kdeplot(
                    contrast_vals, fill=True, color=color, alpha=0.4,
                    linewidth=2, label=label, ax=ax, zorder=3
                )

                hdi_low = record['HDI_95_Low']
                hdi_high = record['HDI_95_High']
                y_offset -= 0.15
                ax.plot(
                    [hdi_low, hdi_high], [y_offset, y_offset],
                    color=color, linewidth=4, solid_capstyle='round',
                    zorder=4
                )
                ax.plot(
                    [record['Physical_Contrast_Median']], [y_offset],
                    marker='|', color=color, markersize=12,
                    markeredgewidth=2, zorder=5
                )

            ax.set_title(
                f"Physical Treatment Effects: {family.upper()} "
                f"vs. {self.labels[self.baseline]}",
                fontsize=14, fontweight='bold'
            )
            ax.set_xlabel(
                f"Physical Shift in {family.upper()} "
                f"(\u0394 from baseline)", fontsize=12
            )
            ax.set_ylabel("Density", fontsize=12)
            ax.legend(loc='upper right', fontsize=10)
            sns.despine(trim=True)

            fig.tight_layout()
            fig.savefig(
                PATHS['figures_supp'] /
                f"contrast_kde_physical_{family}_{self.winning_model}.pdf",
                dpi=300, bbox_inches='tight'
            )
            plt.close(fig)
            self.logger.info(
                f"  Contrast KDE saved: "
                f"contrast_kde_physical_{family}_{self.winning_model}.pdf"
            )

    # -----------------------------------------------------------------
    # MASTER PIPELINE
    # -----------------------------------------------------------------
    def run(self):
        """Orchestrates the full inference and visualization pipeline."""
        # 1. Compute all inferential statistics
        self.compute_all_posterior_statistics()

        # 2. ArviZ native posterior + ROPE plots (Dynamic InferenceData)
        self.render_arviz_posterior_rope()

        # 3. Raincloud plots per parameter family
        model_lower = self.winning_model.lower()
        families = [p for p in ['v', 'a', 't', 'z'] if p in model_lower]
        for family in families:
            self.render_raincloud(family)

        # 4. Supplementary contrast KDE plots
        self.render_contrast_kde()

        # Cleanup
        del self.infdata
        self.infdata = None
        self.physical_contrasts_dict.clear()
        gc.collect()

        self.logger.info(f"\n{'='*70}")
        self.logger.info(
            f"STEP 5 COMPLETE: Inference and visualization exported "
            f"for [{self.winning_model.upper()}]."
        )
        self.logger.info(f"{'='*70}")


# =============================================================================
# PIPELINE EXECUTION
# =============================================================================
if __name__ == "__main__":
    engine = BayesianInferenceVisualizer()
    engine.run()

# Step 6: Data Informed Group-Level Parameter Recovery

In [ ]:
# -*- coding: utf-8 -*-
"""
=============================================================================
STEP 6: Dual-Track Parameter Recovery Validation
=============================================================================
Description:
  - Validates structural identifiability of the optimal model architecture
    via two complementary forward-simulation-then-refit paradigms:
    1. Posterior-Anchored: Extracts pseudo-truth from the empirical joint
       posterior (checks self-consistency of estimation).
    2. Prior-Predictive: Samples pseudo-truth from ecologically valid
       Informative Priors (stress-tests identifiability under unknown but
       plausible ground truth).
  - Generates synthetic datasets preserving the empirical trial structure
    (subjects x conditions x trial counts) using HDDM's built-in
    data generator with proper inverse link functions.
  - Refits synthetic data using dockerHDDM's native parallel chain
    execution (model.sample(chains=N, return_infdata=True)).
  - Computes granular recovery metrics: Bias, RMSE, Coverage, HDI Width.
  - Renders publication-ready diagnostics: Identity Scatter plots and
    HDI Interval Coverage plots.
=============================================================================
"""

import os
import gc
import logging
import traceback
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import arviz as az
import hddm
from hddm.generate import gen_rand_data

# Imported logit to compute correct linear predictor representations
from scipy.special import expit, logit
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import seaborn as sns

# -----------------------------------------------------------------------------
# CONFIGURATION IMPORT & ENVIRONMENT SETUP
# -----------------------------------------------------------------------------
try:
    from hddm_config import (
        CFG, validate_pipeline_lineage, identify_winning_model
    )
except ImportError:
    raise ImportError(
        "CRITICAL ERROR: 'hddm_config.py' not found. "
        "Execute Step 2a to generate the global configuration SSOT."
    )

PATHS = CFG.initialize_directories()

# Academic aesthetics
sns.set_theme(style="ticks")
plt.rcParams.update({
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
    'pdf.fonttype': 42,
    'ps.fonttype': 42
})


# =============================================================================
# LOGGING SETUP
# =============================================================================
def _setup_recovery_logger() -> logging.Logger:
    """Initializes dual-sink logging for the recovery module."""
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    logger = logging.getLogger(f'hddm_recovery_{ts}')
    logger.handlers = []
    logger.setLevel(logging.INFO)

    fmt = logging.Formatter(
        '%(asctime)s - %(levelname)s - %(message)s', datefmt='%H:%M:%S'
    )
    fh = logging.FileHandler(
        PATHS['audit'] / f'dual_recovery_{ts}.log', encoding='utf-8'
    )
    ch = logging.StreamHandler()
    fh.setFormatter(fmt)
    ch.setFormatter(fmt)

    logger.addHandler(fh)
    logger.addHandler(ch)
    return logger


# =============================================================================
# DYNAMIC REGRESSOR FACTORY
# =============================================================================
def _build_regressors(model_name: str, baseline: str) -> List[str]:
    """Constructs Patsy Treatment-coded formulas matching Step 2b topology."""
    regressors = []
    name_lower = model_name.lower()
    if name_lower == 'null':
        return regressors
    for param in ['v', 'a', 'z', 't']:
        if param in name_lower:
            regressors.append(
                f"{param} ~ C(emotion, Treatment('{baseline}'))"
            )
    return regressors


# =============================================================================
# CORE CLASS: DUAL-TRACK RECOVERY ENGINE
# =============================================================================
class DualParameterRecoveryEngine:
    """
    Orchestrates Posterior-Anchored and Prior-Predictive identifiability
    checks to validate the structural recoverability of the optimal
    HDDM architecture selected in Step 4.
    """

    def __init__(self, empirical_data_path: str = 'hddm_data_unfair.csv'):
        self.logger = _setup_recovery_logger()
        self.active_hash = validate_pipeline_lineage(PATHS, self.logger)
        self.winning_model = identify_winning_model(PATHS)
        self.empirical_data_path = empirical_data_path

        # Recovery MCMC hyperparameters
        self.recovery_chains = CFG.n_chains if CFG.run_mode != 'debug' else 2
        self.recovery_samples = 3000 if CFG.run_mode != 'debug' else 500
        self.recovery_burn = 1000 if CFG.run_mode != 'debug' else 100

        self.logger.info("=" * 70)
        self.logger.info(
            f"DUAL-TRACK PARAMETER RECOVERY INITIATED "
            f"(Mode: {CFG.run_mode.upper()})"
        )
        self.logger.info(f"Lineage Hash: {self.active_hash[:24]}...")
        self.logger.info(f"Target Model: [{self.winning_model.upper()}]")
        self.logger.info(
            f"Recovery Protocol: {self.recovery_samples} samples, "
            f"{self.recovery_burn} burn, {self.recovery_chains} chains"
        )
        self.logger.info("=" * 70)

    # -----------------------------------------------------------------
    # LINK FUNCTION & PHYSICAL BOUNDS
    # -----------------------------------------------------------------
    @staticmethod
    def _apply_link_and_clip(param_family: str, lp_val: float) -> float:
        """
        Applies the requisite inverse link function for HDDM regression
        parameters and imposes physiologically plausible bounds.
        """
        if param_family == 'v':
            return float(np.clip(lp_val, -8.0, 8.0))
        elif param_family == 'a':
            return float(np.clip(np.exp(lp_val), 0.2, 4.0))
        elif param_family == 't':
            return float(np.clip(np.exp(lp_val), 0.05, 2.0))
        elif param_family == 'z':
            return float(np.clip(expit(lp_val), 0.05, 0.95))
        return lp_val

    # -----------------------------------------------------------------
    # GROUND TRUTH ESTABLISHMENT
    # -----------------------------------------------------------------
    def establish_ground_truth(self, mode: str) -> Dict[str, float]:
        """
        Generates pseudo-truth target parameters.
        Corrected to sample from Informative distributions on the PHYSICAL
        scale, then inverse-transformed to the Linear Predictor (LP) scale.
        """
        self.logger.info(
            f"[{mode.upper()}] Establishing Ground Truth..."
        )
        pseudo_truth = {}

        if mode == 'posterior':
            nc_path = PATHS['models'] / f"hddm_{self.winning_model}.nc"
            if not nc_path.exists():
                raise FileNotFoundError(f"InferenceData not found: {nc_path.name}")

            infdata = az.from_netcdf(str(nc_path))
            summary = az.summary(infdata, round_to=4)

            focal_indices = [
                idx for idx in summary.index
                if not idx.startswith(('wfpt', 'mc_', '__'))
                and ('Intercept' in idx or 'Treatment' in idx
                     or 'C(emotion' in idx)
            ]
            pseudo_truth = summary.loc[focal_indices, 'mean'].to_dict()
            del infdata, summary
            gc.collect()

        elif mode == 'prior':
            np.random.seed(CFG.base_seed + 999)
            model_lower = self.winning_model.lower()

            for param_family in ['v', 'a', 't', 'z']:
                is_varying = param_family in model_lower
                lp_int = 0.0

                # 1. Sample intercept on PHYSICAL scale, then transform to LP scale
                if param_family == 'v':
                    # Drift rate: Normal(1.0, 1.5) -> Identity link
                    phys_val = stats.norm.rvs(loc=1.0, scale=1.5)
                    lp_int = phys_val
                elif param_family == 'a':
                    # Threshold: Gamma(mu=1.5, std=0.5) -> Log link
                    phys_val = stats.gamma.rvs(a=9.0, scale=0.166)
                    phys_val = np.clip(phys_val, 0.5, 3.5)
                    lp_int = np.log(phys_val)
                elif param_family == 't':
                    # NDT: Truncated Normal [0.1, 0.5] -> Log link
                    a_trunc, b_trunc = (0.1 - 0.3) / 0.1, (0.5 - 0.3) / 0.1
                    phys_val = stats.truncnorm.rvs(a=a_trunc, b=b_trunc, loc=0.3, scale=0.1)
                    phys_val = np.clip(phys_val, 0.05, 0.8)
                    lp_int = np.log(phys_val)
                elif param_family == 'z':
                    # Bias: Beta(5, 5) -> Logit link
                    phys_val = stats.beta.rvs(a=5, b=5)
                    phys_val = np.clip(phys_val, 0.1, 0.9)
                    lp_int = logit(phys_val)

                pseudo_truth[f"{param_family}_Intercept"] = float(lp_int)

                # 2. Sample condition effects (Treatment contrasts on LP scale)
                if is_varying:
                    for emo in CFG.emotion_order:
                        if emo == CFG.baseline_condition:
                            continue
                        
                        effect_key = (
                            f"{param_family}_C(emotion, "
                            f"Treatment('{CFG.baseline_condition}'))"
                            f"[T.{emo}]"
                        )
                        
                        # Effects are deviations on the linear predictor scale
                        if param_family == 'v':
                            eff_val = stats.norm.rvs(loc=0, scale=0.5)
                        elif param_family in ['a', 'z']:
                            eff_val = stats.norm.rvs(loc=0, scale=0.1)
                        elif param_family == 't':
                            eff_val = stats.norm.rvs(loc=0, scale=0.05)
                        else:
                            eff_val = 0.0
                            
                        pseudo_truth[effect_key] = float(eff_val)

        # Export for auditing
        pd.Series(pseudo_truth, name='True_Value').to_csv(
            PATHS['recovery'] / f"ground_truth_dict_mode_{mode}.csv"
        )
        self.logger.info(
            f"  Ground truth: {len(pseudo_truth)} parameters established."
        )
        return pseudo_truth

    # -----------------------------------------------------------------
    # SYNTHETIC DATA GENERATION
    # -----------------------------------------------------------------
    def generate_trialwise_skeleton(
        self, mode: str, truth_dict: Dict[str, float]
    ) -> str:
        """
        Builds synthetic datasets preserving the original empirical trial
        structure (subjects x conditions x trial counts).
        """
        self.logger.info(
            f"[{mode.upper()}] Generating trial-skeleton synthetic data..."
        )

        if not os.path.exists(self.empirical_data_path):
            raise FileNotFoundError(
                f"Empirical data not found: {self.empirical_data_path}"
            )

        emp_df = pd.read_csv(self.empirical_data_path)
        condition_log = []
        trial_rows = []

        seed_offset = 777 if mode == 'posterior' else 888
        np.random.seed(CFG.base_seed + seed_offset)

        for (subj, emo), group in emp_df.groupby(['subj_idx', 'emotion']):
            n_trials = len(group)
            phys_params = {}

            for p in ['v', 'a', 't', 'z']:
                lp_val = truth_dict.get(f"{p}_Intercept", 0.0)
                if emo != CFG.baseline_condition:
                    effect_key = (
                        f"{p}_C(emotion, "
                        f"Treatment('{CFG.baseline_condition}'))"
                        f"[T.{emo}]"
                    )
                    if effect_key not in truth_dict:
                        effect_key = f"{p}_C(emotion)[T.{emo}]"
                    lp_val += truth_dict.get(effect_key, 0.0)

                phys_params[p] = self._apply_link_and_clip(p, lp_val)

            cond_entry = {
                'subj_idx': subj, 'emotion': emo, 'n_trials': n_trials
            }
            cond_entry.update(
                {f"{k}_true": v for k, v in phys_params.items()}
            )
            condition_log.append(cond_entry)

            sim_res = gen_rand_data(phys_params, size=n_trials)
            sim_df = sim_res[0] if isinstance(sim_res, tuple) else sim_res

            group_copy = group.copy().reset_index(drop=True)
            group_copy['rt_sim'] = sim_df['rt'].values
            group_copy['response_sim'] = sim_df['response'].values
            for p in ['v', 'a', 't', 'z']:
                group_copy[f'{p}_true'] = phys_params[p]

            trial_rows.append(group_copy)

        pd.DataFrame(condition_log).to_csv(
            PATHS['recovery'] /
            f"recovery_conditionwise_true_params_{mode}.csv",
            index=False
        )

        final_df = pd.concat(trial_rows, ignore_index=True)
        final_df = final_df.rename(columns={
            'rt': 'rt_emp', 'response': 'response_emp',
            'rt_sim': 'rt', 'response_sim': 'response'
        })

        synth_path = str(
            PATHS['recovery'] / f"recovery_synthetic_data_{mode}.csv"
        )
        final_df.to_csv(synth_path, index=False)
        self.logger.info(
            f"  Synthetic cohort: {len(final_df)} trials generated."
        )
        return synth_path

    # -----------------------------------------------------------------
    # RECOVERY REFIT (dockerHDDM native)
    # -----------------------------------------------------------------
    def execute_recovery_refit(self, mode: str, synth_path: str):
        """Refits the synthetic data using dockerHDDM's native parallel interface."""
        self.logger.info(
            f"\n[{mode.upper()}] Refitting via dockerHDDM "
            f"({self.recovery_chains} chains)..."
        )

        df_synth = pd.read_csv(synth_path)
        df_synth['subj_idx'] = df_synth['subj_idx'].astype(str)

        regressors = _build_regressors(
            self.winning_model, CFG.baseline_condition
        )
        save_prefix = str(
            PATHS['recovery'] / f"recovery_{self.winning_model}_{mode}"
        )

        if not regressors:
            model = hddm.HDDM(
                df_synth,
                include=CFG.include_params,
                is_group_model=True,
                informative=CFG.use_informative_priors,
                p_outlier=CFG.p_outlier
            )
        else:
            model = hddm.HDDMRegressor(
                df_synth,
                regressors,
                include=CFG.include_params,
                group_only_regressors=CFG.group_only_regressors,
                keep_regressor_trace=CFG.keep_regressor_trace,
                informative=CFG.use_informative_priors,
                p_outlier=CFG.p_outlier
            )

        infdata = model.sample(
            self.recovery_samples,
            burn=self.recovery_burn,
            thin=1,
            chains=self.recovery_chains,
            return_infdata=True,
            loglike=False,
            ppc=False,
            save_name=save_prefix
        )

        rec_summary = az.summary(infdata, round_to=4, hdi_prob=0.94)
        rec_summary.to_csv(
            PATHS['recovery'] /
            f"recovery_fit_summary_{self.winning_model}_{mode}.csv"
        )

        self.logger.info("  Recovery fit complete. Summary exported.")
        del model, infdata
        gc.collect()

    # -----------------------------------------------------------------
    # RECOVERY METRICS COMPUTATION
    # -----------------------------------------------------------------
    def calculate_recovery_metrics(
        self, mode: str, truth_dict: Dict[str, float]
    ) -> pd.DataFrame:
        """Computes granular recovery metrics mapping recovered posterior estimates."""
        self.logger.info(f"\n[{mode.upper()}] Computing recovery metrics...")

        summary_path = (
            PATHS['recovery'] /
            f"recovery_fit_summary_{self.winning_model}_{mode}.csv"
        )
        rec_summary = pd.read_csv(summary_path, index_col=0)

        records = []
        for param, gt_val in truth_dict.items():
            if param not in rec_summary.index:
                continue

            rec_row = rec_summary.loc[param]
            rec_mean = float(rec_row['mean'])
            hdi_low = float(rec_row['hdi_3%'])
            hdi_high = float(rec_row['hdi_97%'])

            bias = rec_mean - gt_val
            rel_bias = bias / np.abs(gt_val) if np.abs(gt_val) > 1e-4 else np.nan
            is_covered = hdi_low <= gt_val <= hdi_high

            family = param.split('_')[0] if '_' in param else 'other'
            param_type = 'Intercept' if 'Intercept' in param else 'ConditionEffect'

            records.append({
                'config_hash': self.active_hash,
                'mode': mode,
                'Parameter': param,
                'Family': family,
                'Type': param_type,
                'Ground_Truth': gt_val,
                'Recovered_Mean': rec_mean,
                'Bias': bias,
                'Abs_Bias': np.abs(bias),
                'Relative_Bias': rel_bias,
                'Sq_Error': bias ** 2,
                'HDI_3': hdi_low,
                'HDI_97': hdi_high,
                'HDI_Width': hdi_high - hdi_low,
                'Coverage': int(is_covered)
            })

        df_metrics = pd.DataFrame(records)
        df_metrics.to_csv(
            PATHS['recovery'] / f"recovery_detailed_metrics_{mode}.csv",
            index=False
        )

        if not df_metrics.empty:
            agg = df_metrics.groupby(['Family', 'Type']).agg(
                N=('Parameter', 'count'),
                RMSE=('Sq_Error', lambda x: np.sqrt(x.mean())),
                Mean_Abs_Bias=('Abs_Bias', 'mean'),
                Mean_HDI_Width=('HDI_Width', 'mean'),
                Coverage_Rate=('Coverage', 'mean')
            ).reset_index()
            self.logger.info(f"\n{agg.to_string(index=False)}")

            if len(df_metrics) >= 3:
                r_val, p_val = stats.pearsonr(
                    df_metrics['Ground_Truth'], df_metrics['Recovered_Mean']
                )
                rmse_global = float(np.sqrt(df_metrics['Sq_Error'].mean()))
                self.logger.info(
                    f"\n  Global: r={r_val:.3f} (p={p_val:.2e}), RMSE={rmse_global:.4f}"
                )

        return df_metrics

    # -----------------------------------------------------------------
    # PUBLICATION-READY DIAGNOSTIC VISUALIZATION
    # -----------------------------------------------------------------
    def render_diagnostics(self, mode: str, df_plot: pd.DataFrame):
        """Renders Identity Scatter and HDI Interval Coverage plots."""
        self.logger.info(f"[{mode.upper()}] Rendering diagnostic plots...")
        if df_plot.empty:
            return

        title_prefix = "Posterior-Anchored" if mode == 'posterior' else "Prior-Predictive"
        pal = {'v': '#3C5488', 'a': '#E64B35', 't': '#00A087', 'z': '#8491B4'}

        # --- Plot 1: Identity Scatter per Family ---
        families = df_plot['Family'].unique()
        n_fam = max(1, len(families))
        fig, axes = plt.subplots(1, n_fam, figsize=(5 * n_fam, 5), squeeze=False)
        axes = axes.flatten()

        for ax, fam in zip(axes, families):
            fam_df = df_plot[df_plot['Family'] == fam]
            color = pal.get(fam, '#333333')

            sns.regplot(
                x="Ground_Truth", y="Recovered_Mean", data=fam_df, ax=ax,
                scatter_kws={'alpha': 0.8, 's': 60, 'color': color, 'edgecolors': 'w'},
                line_kws={'color': '#E64B35', 'linewidth': 2}
            )

            all_vals = pd.concat([fam_df['Ground_Truth'], fam_df['Recovered_Mean']])
            margin = max(0.05, (all_vals.max() - all_vals.min()) * 0.1)
            lims = [all_vals.min() - margin, all_vals.max() + margin]
            ax.plot(lims, lims, 'k--', alpha=0.5, zorder=0)
            ax.set_xlim(lims)
            ax.set_ylim(lims)

            if len(fam_df) >= 3:
                r_fam, _ = stats.pearsonr(fam_df['Ground_Truth'], fam_df['Recovered_Mean'])
                rmse_fam = float(np.sqrt(fam_df['Sq_Error'].mean()))
                ax.set_title(f"{fam.upper()} (r={r_fam:.2f}, RMSE={rmse_fam:.3f})", fontweight='bold')
            else:
                ax.set_title(f"{fam.upper()}", fontweight='bold')

            ax.set_xlabel("Pseudo-Truth")
            ax.set_ylabel("Recovered Mean")

        plt.suptitle(f"{title_prefix} Recovery: {self.winning_model.upper()}", y=1.05, fontweight='bold')
        plt.tight_layout()
        plt.savefig(
            PATHS['figures_supp'] / f"recovery_scatter_{self.winning_model}_{mode}.pdf",
            dpi=300, bbox_inches='tight'
        )
        plt.close(fig)

        # --- Plot 2: HDI Interval Coverage Plot ---
        df_sorted = df_plot.sort_values(by=['Family', 'Type'])
        fig, ax = plt.subplots(figsize=(8, max(3, len(df_sorted) * 0.4)))

        for i, (_, row) in enumerate(df_sorted.iterrows()):
            color = pal.get(row['Family'], 'gray')
            ax.plot([row['HDI_3'], row['HDI_97']], [i, i], color=color, linewidth=3, alpha=0.5)
            ax.scatter(row['Recovered_Mean'], i, color=color, s=60, zorder=3)
            ax.scatter(row['Ground_Truth'], i, color='red', marker='x', s=100, linewidths=2, zorder=4)

        ax.set_yticks(range(len(df_sorted)))
        ax.set_yticklabels(df_sorted['Parameter'], fontsize=7)
        ax.set_title(f"{title_prefix}: 94% HDI Interval Coverage", pad=15, fontweight='bold')
        ax.set_xlabel("Parameter Posterior Space")

        red_x = mlines.Line2D([], [], color='red', marker='x', linestyle='None', markersize=8, label='Pseudo-Truth')
        gray_dot = mlines.Line2D([], [], color='gray', marker='o', linestyle='None', markersize=6, label='Recovered Mean + 94% HDI')
        ax.legend(handles=[red_x, gray_dot], bbox_to_anchor=(1.05, 1), loc='upper left')

        plt.tight_layout()
        plt.savefig(
            PATHS['figures_supp'] / f"recovery_coverage_{self.winning_model}_{mode}.pdf",
            dpi=300, bbox_inches='tight'
        )
        plt.close(fig)

        self.logger.info("  Diagnostic plots exported.")

    # -----------------------------------------------------------------
    # MASTER PIPELINE
    # -----------------------------------------------------------------
    def run_full_pipeline(self):
        """Orchestrates both recovery modes sequentially."""
        for mode in ['posterior', 'prior']:
            self.logger.info(f"\n{'*'*60}")
            self.logger.info(f"COMMENCING: {mode.upper()} RECOVERY")
            self.logger.info(f"{'*'*60}")

            truth_dict = self.establish_ground_truth(mode)
            synth_path = self.generate_trialwise_skeleton(mode, truth_dict)
            self.execute_recovery_refit(mode, synth_path)
            metrics_df = self.calculate_recovery_metrics(mode, truth_dict)
            self.render_diagnostics(mode, metrics_df)

        self.logger.info(f"\n{'='*70}")
        self.logger.info("DUAL-TRACK PARAMETER RECOVERY COMPLETED SUCCESSFULLY")
        self.logger.info(f"{'='*70}")


# =============================================================================
# PIPELINE EXECUTION
# =============================================================================
if __name__ == "__main__":
    try:
        engine = DualParameterRecoveryEngine()
        engine.run_full_pipeline()
    except Exception as e:
        print(f"\nCRITICAL PIPELINE FAILURE: {e}")
        traceback.print_exc()